In [1]:
!nvidia-smi

Tue Oct 13 00:18:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|


|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+
|   1  Tesla T4                       Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        

In [2]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

CUDA Available: True
GPU Count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
!python -c "import socket; print(socket.gethostbyname('github.com'))"

140.82.112.3


In [4]:
!git clone https://github.com/Usman-LUMS/small-model-alignment-starter.git /kaggle/working/small-model-alignment-starter

Cloning into '/kaggle/working/small-model-alignment-starter'...


remote: Enumerating objects: 332, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (9/9), done.


remote: Total 332 (delta 11), reused 11 (delta 9), pack-reused 314 (from 1)
Receiving objects: 100% (332/332), 1.07 MiB | 1.97 MiB/s, done.
Resolving deltas: 100% (186/186), done.


In [ ]:
%cd /kaggle/working/small-model-alignment-starter

!pip uninstall -y -q torchao
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps

In [ ]:
!python -m pytest -q tests/test_public_adapter_smoke.py
!git log -1 --oneline

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
).to("cuda:0")

model.eval()

print("Model loaded:", MODEL_NAME)
print("Parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
from datasets import load_dataset

SEED = 42
NUM_PROMPTS = 300

DATA_URL = (
    "https://huggingface.co/datasets/tatsu-lab/alpaca_eval/"
    "resolve/main/alpaca_eval.json"
)

dataset = load_dataset(
    "json",
    data_files=DATA_URL,
    split="train"
)

print("Total examples:", len(dataset))
print("Columns:", dataset.column_names)
print("Example:", dataset[0])

In [ ]:
import json
import random
from pathlib import Path

random.seed(SEED)

indices = random.sample(range(len(dataset)), NUM_PROMPTS)

eval_data = []

for idx in indices:
    example = dataset[idx]

    eval_data.append({
        "id": str(idx),
        "instruction": example["instruction"]
    })

output_dir = Path("outputs/baseline")
output_dir.mkdir(parents=True, exist_ok=True)

eval_path = output_dir / "alpaca_eval_300_seed42.json"

with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(eval_data, f, indent=2, ensure_ascii=False)

print("Evaluation prompts:", len(eval_data))
print("Saved:", eval_path)

In [ ]:
def generate_response(instruction, max_new_tokens=128):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda:0")

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


for example in eval_data[:3]:
    response = generate_response(example["instruction"])

    print("\nINSTRUCTION:", example["instruction"])
    print("RESPONSE:", response)
    print("-" * 70)

In [ ]:
from tqdm.auto import tqdm

baseline_results = []

for example in tqdm(eval_data, desc="Baseline generation"):
    response = generate_response(
        example["instruction"],
        max_new_tokens=128
    )

    baseline_results.append({
        "id": example["id"],
        "instruction": example["instruction"],
        "output": response,
        "generator": "qwen2.5-0.5b-base",
        "dataset": "alpaca_eval"
    })

    # Save progress every 25 examples
    if len(baseline_results) % 25 == 0:
        with open(
            output_dir / "baseline_progress.json",
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                baseline_results,
                f,
                indent=2,
                ensure_ascii=False
            )

print("Generated:", len(baseline_results))

In [ ]:
final_path = output_dir / "qwen_base_alpaca_eval_300.json"

with open(final_path, "w", encoding="utf-8") as f:
    json.dump(
        baseline_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", final_path)
print("Total responses:", len(baseline_results))

assert len(baseline_results) == 300
assert all("output" in item for item in baseline_results)

In [ ]:
for example in baseline_results[:5]:
    print("=" * 80)
    print("Instruction:", example["instruction"])
    print("\nGenerated response:")
    print(example["output"])

In [ ]:
import shutil

archive_path = shutil.make_archive(
    "/kaggle/working/pa2_baseline_outputs",
    "zip",
    root_dir=str(output_dir)
)

print("Archive created:", archive_path)

In [ ]:
import os
import json
import random
import numpy as np
import torch

from pathlib import Path

PROJECT_DIR = Path("/kaggle/working/small-model-alignment-starter")
os.chdir(PROJECT_DIR)

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# First run a small training smoke test.
# Change to "full" after the smoke test succeeds.
RUN_MODE = "smoke"

TRAIN_SIZE = 64 if RUN_MODE == "smoke" else 30000
VAL_SIZE = 32 if RUN_MODE == "smoke" else 1000

MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256

BATCH_SIZE = 2
GRAD_ACCUM = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.1
EPOCHS = 1

OUTPUT_DIR = PROJECT_DIR / "outputs" / f"sft_{RUN_MODE}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Mode:", RUN_MODE)
print("Training examples:", TRAIN_SIZE)
print("Validation examples:", VAL_SIZE)
print("Output:", OUTPUT_DIR)

In [ ]:
from datasets import load_dataset

raw_train = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="train_sft[:50000]"
)

raw_val = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="test_sft[:3000]"
)

def extract_pairs(dataset, limit, seed):
    examples = []

    shuffled = dataset.shuffle(seed=seed)

    for row in shuffled:
        messages = row["messages"]

        if len(messages) < 2:
            continue

        if messages[0]["role"] != "user":
            continue

        if messages[1]["role"] != "assistant":
            continue

        prompt = messages[0]["content"].strip()
        response = messages[1]["content"].strip()

        if prompt and response:
            examples.append({
                "prompt": prompt,
                "response": response
            })

        if len(examples) == limit:
            break

    if len(examples) != limit:
        raise ValueError(
            f"Expected {limit} examples, got {len(examples)}"
        )

    return examples


train_examples = extract_pairs(raw_train, TRAIN_SIZE, SEED)
val_examples = extract_pairs(raw_val, VAL_SIZE, SEED)

print("Train:", len(train_examples))
print("Validation:", len(val_examples))
print("Example instruction:", train_examples[0]["prompt"][:200])
print("Example response:", train_examples[0]["response"][:200])

with open(OUTPUT_DIR / "data_manifest.json", "w") as f:
    json.dump({
        "source": "HuggingFaceH4/ultrachat_200k",
        "train_split": "train_sft",
        "validation_split": "test_sft",
        "seed": SEED,
        "train_examples": len(train_examples),
        "validation_examples": len(val_examples),
        "selection": "first user/assistant pair after seeded shuffle"
    }, f, indent=2)

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_example(example):
    prompt = (
        "### Instruction:\n"
        + example["prompt"]
        + "\n\n### Response:\n"
    )

    prompt_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False
    )[:MAX_PROMPT_LENGTH]

    response_ids = tokenizer.encode(
        example["response"],
        add_special_tokens=False
    )

    if tokenizer.eos_token_id is not None:
        response_ids.append(tokenizer.eos_token_id)

    available = MAX_LENGTH - len(prompt_ids)
    response_ids = response_ids[:available]

    input_ids = prompt_ids + response_ids

    labels = (
        [-100] * len(prompt_ids)
        + response_ids
    )

    return {
        "input_ids": input_ids,
        "labels": labels
    }


class SFTDataset(Dataset):
    def __init__(self, examples):
        self.data = [
            tokenize_example(x)
            for x in examples
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


def collate_fn(batch):
    width = max(len(x["input_ids"]) for x in batch)

    input_ids = []
    labels = []
    attention_masks = []

    for example in batch:
        length = len(example["input_ids"])
        pad = width - length

        input_ids.append(
            example["input_ids"]
            + [tokenizer.pad_token_id] * pad
        )

        labels.append(
            example["labels"] + [-100] * pad
        )

        attention_masks.append(
            [1] * length + [0] * pad
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }


train_dataset = SFTDataset(train_examples)
val_dataset = SFTDataset(val_examples)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

sample = next(iter(train_loader))

print("Input shape:", sample["input_ids"].shape)
print("Labels shape:", sample["labels"].shape)
print("Response tokens:", (sample["labels"] != -100).sum().item())

assert (sample["labels"] != -100).sum().item() > 0
assert sample["input_ids"].shape == sample["labels"].shape

In [ ]:
from transformers import AutoModelForCausalLM

device = torch.device("cuda:0")

# Release previous inference model to save GPU memory.
if "model" in globals():
    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32
)

model.to(device)
model.train()

model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Model:", MODEL_NAME)
print("Device:", next(model.parameters()).device)
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Model dtype:", next(model.parameters()).dtype)

In [ ]:
import math
from transformers import get_cosine_schedule_with_warmup

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

steps_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM
)

TOTAL_STEPS = steps_per_epoch * EPOCHS
WARMUP_STEPS = max(1, int(TOTAL_STEPS * 0.03))

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP_STEPS,
    num_training_steps=TOTAL_STEPS
)

scaler = torch.amp.GradScaler("cuda")

print("Optimizer: AdamW")
print("Total optimizer steps:", TOTAL_STEPS)
print("Warmup steps:", WARMUP_STEPS)
print("Gradient accumulation:", GRAD_ACCUM)

In [ ]:
from tqdm.auto import tqdm
import torch.nn.functional as F

training_history = []
best_val_loss = float("inf")


def compute_batch_loss(batch):
    inputs = {
        key: value.to(device)
        for key, value in batch.items()
    }

    with torch.autocast("cuda", dtype=torch.float16):
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

    # Shift for causal language modeling.
    logits = outputs.logits[:, :-1, :].float()
    labels = inputs["labels"][:, 1:]

    # Only response tokens contribute to loss.
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        labels.reshape(-1),
        ignore_index=-100
    )

    return loss


@torch.no_grad()
def evaluate_sft():
    model.eval()
    losses = []

    for batch in val_loader:
        loss = compute_batch_loss(batch)
        losses.append(loss.item())

    model.train()

    return sum(losses) / len(losses)


def save_checkpoint(name):
    path = OUTPUT_DIR / name
    path.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(
        path,
        safe_serialization=True
    )
    tokenizer.save_pretrained(path)

    print("Checkpoint saved:", path)


def save_metrics(record):
    with open(
        OUTPUT_DIR / "metrics.jsonl",
        "a",
        encoding="utf-8"
    ) as f:
        f.write(json.dumps(record) + "\n")


print("Training functions initialized.")

In [ ]:
import time

start_time = time.time()
global_step = 0
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):

    model.train()
    epoch_losses = []

    progress = tqdm(
        train_loader,
        desc=f"SFT Epoch {epoch + 1}/{EPOCHS}"
    )

    accumulated = 0

    for batch_idx, batch in enumerate(progress):

        loss = compute_batch_loss(batch)
        epoch_losses.append(loss.item())

        scaled_loss = loss / GRAD_ACCUM

        scaler.scale(scaled_loss).backward()
        accumulated += 1

        is_last_batch = batch_idx == len(train_loader) - 1

        if accumulated == GRAD_ACCUM or is_last_batch:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            optimizer.zero_grad(set_to_none=True)
            accumulated = 0

            global_step += 1

            progress.set_postfix(
                loss=f"{loss.item():.4f}",
                step=global_step
            )

            eval_interval = (
                1 if RUN_MODE == "smoke" else 100
            )

            if global_step % eval_interval == 0 or global_step == TOTAL_STEPS:

                val_loss = evaluate_sft()

                record = {
                    "step": global_step,
                    "epoch": epoch + 1,
                    "train_loss": sum(epoch_losses) / len(epoch_losses),
                    "val_loss": val_loss,
                    "lr": scheduler.get_last_lr()[0]
                }

                training_history.append(record)
                save_metrics(record)

                print(
                    f"Step {global_step} | "
                    f"Train: {record['train_loss']:.4f} | "
                    f"Validation: {val_loss:.4f}"
                )

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    save_checkpoint("best")

save_checkpoint("final")

elapsed = (time.time() - start_time) / 60

print("Training complete!")
print("Minutes:", round(elapsed, 2))
print("Best validation loss:", best_val_loss)

In [ ]:
import matplotlib.pyplot as plt

steps = [x["step"] for x in training_history]
train_losses = [x["train_loss"] for x in training_history]
val_losses = [x["val_loss"] for x in training_history]

plt.figure(figsize=(9, 5))
plt.plot(steps, train_losses, label="Train loss")
plt.plot(steps, val_losses, label="Validation loss")

plt.xlabel("Optimizer steps")
plt.ylabel("Loss")
plt.title(f"Qwen2.5-0.5B SFT — {RUN_MODE}")
plt.legend()
plt.grid(True, alpha=0.3)

plt.savefig(
    OUTPUT_DIR / "sft_loss.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
from pathlib import Path

best_path = OUTPUT_DIR / "best"
final_path = OUTPUT_DIR / "final"

print("Best checkpoint exists:", best_path.exists())
print("Final checkpoint exists:", final_path.exists())

assert (best_path / "config.json").exists()
assert (final_path / "config.json").exists()

print("SFT checkpoints verified.")

In [ ]:
import shutil

archive = shutil.make_archive(
    f"/kaggle/working/pa2_sft_{RUN_MODE}",
    "zip",
    root_dir=str(OUTPUT_DIR)
)

print("Archive created:", archive)

In [ ]:
import os
import gc
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import torch

PROJECT_DIR = Path("/kaggle/working/small-model-alignment-starter")
os.chdir(PROJECT_DIR)

SEED = 42
MODEL_NAME = "Qwen/Qwen2.5-0.5B"
DATASET_NAME = "HuggingFaceH4/ultrachat_200k"

TRAIN_SIZE = 30000
VAL_SIZE = 1000

MAX_LENGTH = 512
MAX_PROMPT_LENGTH = 256
BATCH_SIZE = 2
GRAD_ACCUM = 16
EPOCHS = 1
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.1
EVAL_EVERY = 100

DEVICE = torch.device("cuda:0")

FULL_OUTPUT = PROJECT_DIR / "outputs" / "sft_full"
FULL_OUTPUT.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Full SFT configuration ready")
print("Training:", TRAIN_SIZE)
print("Validation:", VAL_SIZE)
print("Output:", FULL_OUTPUT)

In [ ]:
from datasets import load_dataset

raw_train_full = load_dataset(
    DATASET_NAME,
    split="train_sft"
)

raw_val_full = load_dataset(
    DATASET_NAME,
    split="test_sft"
)

def full_extract_pairs(source, size, seed):
    selected = []

    for row in source.shuffle(seed=seed):
        messages = row["messages"]

        if len(messages) < 2:
            continue

        if messages[0]["role"] != "user":
            continue

        if messages[1]["role"] != "assistant":
            continue

        prompt = messages[0]["content"].strip()
        response = messages[1]["content"].strip()

        if prompt and response:
            selected.append({
                "prompt": prompt,
                "response": response
            })

        if len(selected) == size:
            break

    if len(selected) != size:
        raise ValueError(f"Only collected {len(selected)} examples")

    return selected

full_train_examples = full_extract_pairs(
    raw_train_full, TRAIN_SIZE, SEED
)

full_val_examples = full_extract_pairs(
    raw_val_full, VAL_SIZE, SEED
)

print("Train:", len(full_train_examples))
print("Validation:", len(full_val_examples))

assert len(full_train_examples) == 30000
assert len(full_val_examples) == 1000

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

full_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if full_tokenizer.pad_token_id is None:
    full_tokenizer.pad_token = full_tokenizer.eos_token

FULL_TEMPLATE = "### Instruction:\n{instruction}\n\n### Response:\n"

def full_tokenize(example):
    prompt = FULL_TEMPLATE.format(
        instruction=example["prompt"]
    )

    prompt_ids = full_tokenizer.encode(
        prompt, add_special_tokens=False
    )[:MAX_PROMPT_LENGTH]

    response_ids = full_tokenizer.encode(
        example["response"], add_special_tokens=False
    )

    if full_tokenizer.eos_token_id is not None:
        response_ids.append(full_tokenizer.eos_token_id)

    response_ids = response_ids[:MAX_LENGTH - len(prompt_ids)]

    input_ids = prompt_ids + response_ids
    labels = [-100] * len(prompt_ids) + response_ids

    return {"input_ids": input_ids, "labels": labels}


class FullSFTDataset(Dataset):
    def __init__(self, examples):
        self.rows = [full_tokenize(x) for x in examples]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


def full_collate(batch):
    width = max(len(x["input_ids"]) for x in batch)
    ids, labels, masks = [], [], []

    for row in batch:
        n = len(row["input_ids"])
        pad = width - n

        ids.append(
            row["input_ids"] + [full_tokenizer.pad_token_id] * pad
        )
        labels.append(row["labels"] + [-100] * pad)
        masks.append([1] * n + [0] * pad)

    return {
        "input_ids": torch.tensor(ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(masks, dtype=torch.long)
    }


full_train_loader = DataLoader(
    FullSFTDataset(full_train_examples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=full_collate
)

full_val_loader = DataLoader(
    FullSFTDataset(full_val_examples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=full_collate
)

print("Training batches:", len(full_train_loader))
print("Validation batches:", len(full_val_loader))
print("Expected optimizer steps:",
      math.ceil(len(full_train_loader) / GRAD_ACCUM))

In [ ]:
from transformers import AutoModelForCausalLM

# Release models and optimizers from earlier notebook cells.
for name in [
    "model", "sft_model", "optimizer", "scheduler",
    "scaler", "full_model", "full_optimizer",
    "full_scheduler", "full_scaler"
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

full_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True
).to(DEVICE)

full_model.config.use_cache = False
full_model.gradient_checkpointing_enable()
full_model.train()

print("Model:", MODEL_NAME)
print("Device:", next(full_model.parameters()).device)
print("Dtype:", next(full_model.parameters()).dtype)
print(
    "Trainable parameters:",
    sum(p.numel() for p in full_model.parameters() if p.requires_grad)
)

assert next(full_model.parameters()).dtype == torch.float32

In [ ]:
from transformers import get_cosine_schedule_with_warmup

full_optimizer = torch.optim.AdamW(
    full_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

FULL_TOTAL_STEPS = math.ceil(
    len(full_train_loader) / GRAD_ACCUM
) * EPOCHS

FULL_WARMUP_STEPS = max(
    1,
    round(FULL_TOTAL_STEPS * 0.03)
)

full_scheduler = get_cosine_schedule_with_warmup(
    full_optimizer,
    num_warmup_steps=FULL_WARMUP_STEPS,
    num_training_steps=FULL_TOTAL_STEPS
)

full_scaler = torch.amp.GradScaler("cuda")

print("Total optimizer steps:", FULL_TOTAL_STEPS)
print("Warmup steps:", FULL_WARMUP_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUM)

In [ ]:
import torch.nn.functional as F

def full_compute_loss(batch):
    input_ids = batch["input_ids"].to(DEVICE)
    attention_mask = batch["attention_mask"].to(DEVICE)
    labels = batch["labels"].to(DEVICE)

    with torch.autocast("cuda", dtype=torch.float16):
        outputs = full_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False
        )

    shifted_logits = outputs.logits[:, :-1, :].float()
    shifted_labels = labels[:, 1:]

    return F.cross_entropy(
        shifted_logits.reshape(-1, shifted_logits.size(-1)),
        shifted_labels.reshape(-1),
        ignore_index=-100,
        reduction="mean"
    )


@torch.no_grad()
def full_evaluate():
    full_model.eval()

    total_loss = 0.0
    total_tokens = 0

    for batch in full_val_loader:
        loss = full_compute_loss(batch)

        tokens = (
            batch["labels"][:, 1:] != -100
        ).sum().item()

        total_loss += loss.item() * tokens
        total_tokens += tokens

    full_model.train()

    return total_loss / max(total_tokens, 1)


def full_save_checkpoint(name):
    path = FULL_OUTPUT / name
    path.mkdir(parents=True, exist_ok=True)

    full_model.save_pretrained(
        path,
        safe_serialization=True
    )

    full_tokenizer.save_pretrained(path)

    print("Saved:", path)


def full_save_metric(record):
    with open(
        FULL_OUTPUT / "metrics.jsonl",
        "a",
        encoding="utf-8"
    ) as f:
        f.write(json.dumps(record) + "\n")


print("Full SFT functions ready.")

In [ ]:
full_optimizer.zero_grad(set_to_none=True)
full_model.train()

test_batch = next(iter(full_train_loader))

test_loss = full_compute_loss(test_batch)

print("Initial loss:", test_loss.item())

assert torch.isfinite(test_loss).item(), "Non-finite loss"

full_scaler.scale(test_loss).backward()

has_gradients = any(
    p.grad is not None
    for p in full_model.parameters()
    if p.requires_grad
)

print("Gradients available:", has_gradients)

assert has_gradients

full_optimizer.zero_grad(set_to_none=True)
torch.cuda.empty_cache()

print("Full SFT sanity check passed.")

In [ ]:
full_optimizer.zero_grad(set_to_none=True)
full_model.train()

test_batch = next(iter(full_train_loader))

test_loss = full_compute_loss(test_batch)

print("Initial loss:", test_loss.item())

assert torch.isfinite(test_loss).item(), "Non-finite loss"

full_scaler.scale(test_loss).backward()

has_gradients = any(
    p.grad is not None
    for p in full_model.parameters()
    if p.requires_grad
)

print("Gradients available:", has_gradients)

assert has_gradients

full_optimizer.zero_grad(set_to_none=True)
torch.cuda.empty_cache()

print("Full SFT sanity check passed.")

In [ ]:
from tqdm.auto import tqdm

full_metrics_path = FULL_OUTPUT / "metrics.jsonl"

if full_metrics_path.exists():
    raise RuntimeError(
        "Full SFT metrics already exist. "
        "Check the previous run before starting again."
    )

full_history = []
full_best_val = float("inf")
full_global_step = 0

full_start = time.time()
full_optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    full_model.train()

    interval_loss_sum = 0.0
    interval_token_count = 0

    progress = tqdm(
        full_train_loader,
        desc=f"Full SFT Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch_idx, batch in enumerate(progress):

        # Actual accumulation window size,
        # including the smaller final window.
        window_start = (
            batch_idx // GRAD_ACCUM
        ) * GRAD_ACCUM

        window_end = min(
            window_start + GRAD_ACCUM,
            len(full_train_loader)
        )

        window_size = window_end - window_start

        loss = full_compute_loss(batch)

        if not torch.isfinite(loss).item():
            raise RuntimeError(
                f"Non-finite loss at batch {batch_idx}"
            )

        tokens = (
            batch["labels"][:, 1:] != -100
        ).sum().item()

        interval_loss_sum += loss.item() * tokens
        interval_token_count += tokens

        full_scaler.scale(
            loss / window_size
        ).backward()

        if batch_idx + 1 == window_end:

            full_scaler.unscale_(full_optimizer)

            torch.nn.utils.clip_grad_norm_(
                full_model.parameters(),
                max_norm=1.0
            )

            previous_scale = full_scaler.get_scale()

            full_scaler.step(full_optimizer)
            full_scaler.update()

            updated = (
                full_scaler.get_scale() >= previous_scale
            )

            if updated:
                full_scheduler.step()
                full_global_step += 1

            full_optimizer.zero_grad(set_to_none=True)

            progress.set_postfix(
                loss=f"{loss.item():.4f}",
                step=full_global_step
            )

            should_evaluate = updated and (
                full_global_step % EVAL_EVERY == 0
                or (
                    epoch == EPOCHS - 1
                    and batch_idx + 1 == len(full_train_loader)
                )
            )

            if should_evaluate:
                val_loss = full_evaluate()

                train_loss = (
                    interval_loss_sum
                    / max(interval_token_count, 1)
                )

                record = {
                    "step": full_global_step,
                    "epoch": epoch + 1,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "learning_rate": full_scheduler.get_last_lr()[0],
                    "elapsed_minutes": (
                        time.time() - full_start
                    ) / 60
                }

                full_history.append(record)
                full_save_metric(record)

                print(
                    f"\nStep {full_global_step}/{FULL_TOTAL_STEPS}"
                    f" | Train loss: {train_loss:.4f}"
                    f" | Validation loss: {val_loss:.4f}"
                )

                interval_loss_sum = 0.0
                interval_token_count = 0

                if val_loss < full_best_val:
                    full_best_val = val_loss
                    full_save_checkpoint("best")

full_save_checkpoint("final")

print("\nFULL SFT TRAINING COMPLETED")
print("Optimizer steps:", full_global_step)
print("Best validation loss:", full_best_val)
print(
    "Elapsed minutes:",
    round((time.time() - full_start) / 60, 2)
)

In [ ]:
import matplotlib.pyplot as plt

assert full_history, "No training metrics recorded."

steps = [r["step"] for r in full_history]
train_losses = [r["train_loss"] for r in full_history]
val_losses = [r["val_loss"] for r in full_history]

plt.figure(figsize=(10, 5))

plt.plot(
    steps, train_losses,
    marker="o",
    label="Training loss"
)

plt.plot(
    steps, val_losses,
    marker="s",
    label="Validation loss"
)

plt.xlabel("Optimizer steps")
plt.ylabel("Cross-entropy loss")
plt.title("Qwen2.5-0.5B — Full SFT Training")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plot_path = FULL_OUTPUT / "sft_loss.png"
plt.savefig(plot_path, dpi=150)
plt.show()

print("Saved plot:", plot_path)

In [ ]:
full_best_path = FULL_OUTPUT / "best"
full_final_path = FULL_OUTPUT / "final"

for path in [full_best_path, full_final_path]:
    print("Checking:", path)

    assert (path / "config.json").exists()

    weights = (
        list(path.glob("*.safetensors"))
        + list(path.glob("*.bin"))
    )

    assert weights, f"No model weights found: {path}"

    print("Verified:", path.name)

print("Both full SFT checkpoints verified.")

In [ ]:
from transformers import AutoModelForCausalLM

# Free training model memory.
del full_model
del full_optimizer
del full_scheduler
del full_scaler

gc.collect()
torch.cuda.empty_cache()

full_sft_inference_model = AutoModelForCausalLM.from_pretrained(
    str(full_best_path),
    torch_dtype=torch.float16
).to(DEVICE)

full_sft_inference_model.eval()

instruction = "Explain supervised fine-tuning in simple terms."

prompt = FULL_TEMPLATE.format(instruction=instruction)

inputs = full_tokenizer(
    prompt,
    return_tensors="pt"
).to(DEVICE)

with torch.inference_mode():
    generated = full_sft_inference_model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=full_tokenizer.pad_token_id,
        eos_token_id=full_tokenizer.eos_token_id
    )

response_tokens = generated[0][inputs["input_ids"].shape[1]:]

response = full_tokenizer.decode(
    response_tokens,
    skip_special_tokens=True
).strip()

print("INSTRUCTION:")
print(instruction)

print("\nSFT RESPONSE:")
print(response)

In [ ]:
import shutil

# Release the inference model.
del full_sft_inference_model
gc.collect()
torch.cuda.empty_cache()

archive_path = shutil.make_archive(
    "/kaggle/working/pa2_sft_full",
    "zip",
    root_dir=str(FULL_OUTPUT)
)

print("Archive:", archive_path)
print("Archive size (GB):", round(
    Path(archive_path).stat().st_size / (1024 ** 3), 2
))

In [ ]:
from pathlib import Path
import json
import math

print("FULL SFT STATUS")
print("-" * 45)

expected_windows = math.ceil(
    len(full_train_loader) / GRAD_ACCUM
) * EPOCHS

print("Training examples:", TRAIN_SIZE)
print("Validation examples:", VAL_SIZE)
print("Completed optimizer steps:", full_global_step)
print("Planned accumulation windows:", expected_windows)
print("Skipped optimizer updates:",
      expected_windows - full_global_step)
print("Best validation loss:", full_best_val)
print("Best checkpoint:", full_best_path)
print("Final checkpoint:", full_final_path)
print("Training plot:", plot_path)
print("Archive:", archive_path)

assert full_global_step <= expected_windows
assert full_global_step > 0

assert (full_best_path / "config.json").exists()
assert (full_final_path / "config.json").exists()

assert plot_path.exists()
assert Path(archive_path).exists()

with open(FULL_OUTPUT / "training_summary.json", "w") as f:
    json.dump({
        "train_examples": TRAIN_SIZE,
        "validation_examples": VAL_SIZE,
        "planned_optimizer_steps": expected_windows,
        "successful_optimizer_steps": full_global_step,
        "skipped_optimizer_updates": expected_windows - full_global_step,
        "best_validation_loss": full_best_val,
        "status": "completed_with_skipped_update"
    }, f, indent=2)

print("\nSFT ARTIFACT VERIFICATION PASSED")

In [ ]:
import os
import gc
import json
import math
import random
import time

from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

PROJECT_DIR = Path(
    "/kaggle/working/small-model-alignment-starter"
)
os.chdir(PROJECT_DIR)

SFT_CHECKPOINT = PROJECT_DIR / "outputs/sft_full/best"

assert (SFT_CHECKPOINT / "config.json").exists(), (
    "Best SFT checkpoint not found. "
    "Restore your saved SFT output before continuing."
)

DPO_ROOT = PROJECT_DIR / "outputs/dpo"
DPO_ROOT.mkdir(parents=True, exist_ok=True)

DPO_SEED = 42
DPO_TRAIN_SIZE = 15000
DPO_VAL_SIZE = 1000

DPO_MAX_LENGTH = 512
DPO_MAX_PROMPT_LENGTH = 256

DPO_BATCH_SIZE = 1
DPO_GRAD_ACCUM = 32
DPO_EPOCHS = 1

DPO_LR = 1e-6
DPO_BETA = 0.1
DPO_GRAD_CLIP = 1.0

DPO_EVAL_EVERY = 250

POLICY_DEVICE = torch.device("cuda:0")
REFERENCE_DEVICE = torch.device("cuda:1")

assert torch.cuda.device_count() >= 2, (
    "Phase 5 requires two accessible GPUs "
    "for this implementation."
)

random.seed(DPO_SEED)
np.random.seed(DPO_SEED)
torch.manual_seed(DPO_SEED)
torch.cuda.manual_seed_all(DPO_SEED)

dpo_tokenizer = AutoTokenizer.from_pretrained(
    str(SFT_CHECKPOINT)
)

if dpo_tokenizer.pad_token_id is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

DPO_PROMPT_TEMPLATE = (
    "### Instruction:\n{instruction}\n\n### Response:\n"
)

print("DPO configuration initialized.")
print("SFT checkpoint:", SFT_CHECKPOINT)
print("GPUs:", torch.cuda.device_count())
print("Beta:", DPO_BETA)
print("Training pairs:", DPO_TRAIN_SIZE)
print("Validation pairs:", DPO_VAL_SIZE)

In [ ]:
print("Tokenizer class:", type(dpo_tokenizer).__name__)
print("Tokenizer name:", dpo_tokenizer.name_or_path)
print("Vocabulary size:", len(dpo_tokenizer))

print("EOS token:", dpo_tokenizer.eos_token)
print("PAD token:", dpo_tokenizer.pad_token)

print("Test tokens:")
print(dpo_tokenizer.tokenize("Hello, how are you?"))

In [ ]:
def parse_hh_pair(row):
    chosen_text = row["chosen"]
    rejected_text = row["rejected"]

    separator = "\n\nAssistant:"

    if separator not in chosen_text:
        return None

    if separator not in rejected_text:
        return None

    chosen_prefix, chosen_response = chosen_text.rsplit(
        separator, 1
    )

    rejected_prefix, rejected_response = rejected_text.rsplit(
        separator, 1
    )

    if chosen_prefix != rejected_prefix:
        return None

    # Single-turn pairs only.
    # Preserve the final user instruction without
    # introducing previous assistant turns.
    if chosen_prefix.count("\n\nHuman:") != 1:
        return None

    if not chosen_prefix.startswith("\n\nHuman:"):
        return None

    prompt = chosen_prefix[len("\n\nHuman:"):].strip()
    chosen = chosen_response.strip()
    rejected = rejected_response.strip()

    if not prompt or not chosen or not rejected:
        return None

    if chosen == rejected:
        return None

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected
    }


def parse_ultrafeedback_pair(row):
    chosen_messages = row["chosen"]
    rejected_messages = row["rejected"]

    if len(chosen_messages) != 2:
        return None

    if len(rejected_messages) != 2:
        return None

    if chosen_messages[0]["role"] != "user":
        return None

    if rejected_messages[0]["role"] != "user":
        return None

    if chosen_messages[1]["role"] != "assistant":
        return None

    if rejected_messages[1]["role"] != "assistant":
        return None

    prompt = chosen_messages[0]["content"].strip()

    if prompt != rejected_messages[0]["content"].strip():
        return None

    chosen = chosen_messages[1]["content"].strip()
    rejected = rejected_messages[1]["content"].strip()

    if not prompt or not chosen or not rejected:
        return None

    if chosen == rejected:
        return None

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected
    }


def select_pairs(source, parser, count, seed):
    pairs = []

    for row in source.shuffle(seed=seed):
        result = parser(row)

        if result is not None:
            pairs.append(result)

        if len(pairs) == count:
            break

    if len(pairs) != count:
        raise RuntimeError(
            f"Required {count} pairs, found {len(pairs)}."
        )

    return pairs


print("Loading HH-RLHF...")

hh_train_raw = load_dataset(
    "Anthropic/hh-rlhf",
    split="train"
)

hh_val_raw = load_dataset(
    "Anthropic/hh-rlhf",
    split="test"
)

hh_train = select_pairs(
    hh_train_raw, parse_hh_pair,
    DPO_TRAIN_SIZE, DPO_SEED
)

hh_val = select_pairs(
    hh_val_raw, parse_hh_pair,
    DPO_VAL_SIZE, DPO_SEED
)

print("HH train:", len(hh_train))
print("HH validation:", len(hh_val))


print("\nLoading UltraFeedback...")

uf_train_raw = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs"
)

uf_val_raw = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="test_prefs"
)

uf_train = select_pairs(
    uf_train_raw, parse_ultrafeedback_pair,
    DPO_TRAIN_SIZE, DPO_SEED
)

uf_val = select_pairs(
    uf_val_raw, parse_ultrafeedback_pair,
    DPO_VAL_SIZE, DPO_SEED
)

print("UltraFeedback train:", len(uf_train))
print("UltraFeedback validation:", len(uf_val))

assert len(hh_train) == 15000
assert len(hh_val) == 1000
assert len(uf_train) == 15000
assert len(uf_val) == 1000

print("\nBoth preference datasets prepared.")

In [ ]:
def encode_dpo_pair(example):
    prompt_text = DPO_PROMPT_TEMPLATE.format(
        instruction=example["prompt"]
    )

    prompt_ids = dpo_tokenizer.encode(
        prompt_text,
        add_special_tokens=False
    )[:DPO_MAX_PROMPT_LENGTH]

    def encode_response(text):
        response_ids = dpo_tokenizer.encode(
            text,
            add_special_tokens=False
        )

        if dpo_tokenizer.eos_token_id is not None:
            response_ids.append(
                dpo_tokenizer.eos_token_id
            )

        available = DPO_MAX_LENGTH - len(prompt_ids)
        response_ids = response_ids[:available]

        if not response_ids:
            raise ValueError("Empty response after truncation.")

        return {
            "input_ids": prompt_ids + response_ids,
            "labels": (
                [-100] * len(prompt_ids)
                + response_ids
            )
        }

    return {
        "chosen": encode_response(example["chosen"]),
        "rejected": encode_response(example["rejected"])
    }


class DPODataset(Dataset):
    def __init__(self, examples):
        self.rows = [
            encode_dpo_pair(example)
            for example in examples
        ]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.rows[index]


def pad_dpo_sequences(rows):
    width = max(len(row["input_ids"]) for row in rows)

    input_ids = []
    labels = []
    attention_masks = []

    for row in rows:
        length = len(row["input_ids"])
        padding = width - length

        input_ids.append(
            row["input_ids"]
            + [dpo_tokenizer.pad_token_id] * padding
        )

        labels.append(
            row["labels"] + [-100] * padding
        )

        attention_masks.append(
            [1] * length + [0] * padding
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(
            attention_masks, dtype=torch.long
        ),
        "labels": torch.tensor(labels, dtype=torch.long)
    }


def dpo_collate(batch):
    return {
        "chosen": pad_dpo_sequences([
            row["chosen"] for row in batch
        ]),
        "rejected": pad_dpo_sequences([
            row["rejected"] for row in batch
        ])
    }


print("DPO dataset and batching functions ready.")

In [ ]:
def dpo_response_logps(model, batch, device, trainable):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    with torch.set_grad_enabled(trainable):
        with torch.autocast(
            "cuda", dtype=torch.float16
        ):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False
            )

        logits = outputs.logits[:, :-1, :].float()
        targets = labels[:, 1:]

        valid = targets != -100
        safe_targets = targets.masked_fill(~valid, 0)

        log_probs = F.log_softmax(logits, dim=-1)

        selected = log_probs.gather(
            -1, safe_targets.unsqueeze(-1)
        ).squeeze(-1)

        selected = selected.masked_fill(~valid, 0.0)

        return selected.sum(dim=-1)


def dpo_batch_metrics(policy, reference, batch, beta):
    policy_chosen = dpo_response_logps(
        policy, batch["chosen"], POLICY_DEVICE, True
    )

    policy_rejected = dpo_response_logps(
        policy, batch["rejected"], POLICY_DEVICE, True
    )

    reference_chosen = dpo_response_logps(
        reference, batch["chosen"],
        REFERENCE_DEVICE, False
    ).to(POLICY_DEVICE)

    reference_rejected = dpo_response_logps(
        reference, batch["rejected"],
        REFERENCE_DEVICE, False
    ).to(POLICY_DEVICE)

    chosen_reward = beta * (
        policy_chosen - reference_chosen
    )

    rejected_reward = beta * (
        policy_rejected - reference_rejected
    )

    margin = chosen_reward - rejected_reward

    loss = -F.logsigmoid(margin).mean()

    accuracy = (
        chosen_reward > rejected_reward
    ).float().mean()

    return {
        "loss": loss,
        "accuracy": accuracy,
        "margin": margin.mean()
    }


print("Manual DPO objective initialized.")

In [ ]:
def dpo_response_logps(model, batch, device, trainable):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    with torch.set_grad_enabled(trainable):
        with torch.autocast(
            "cuda", dtype=torch.float16
        ):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False
            )

        logits = outputs.logits[:, :-1, :].float()
        targets = labels[:, 1:]

        valid = targets != -100
        safe_targets = targets.masked_fill(~valid, 0)

        log_probs = F.log_softmax(logits, dim=-1)

        selected = log_probs.gather(
            -1, safe_targets.unsqueeze(-1)
        ).squeeze(-1)

        selected = selected.masked_fill(~valid, 0.0)

        return selected.sum(dim=-1)


def dpo_batch_metrics(policy, reference, batch, beta):
    policy_chosen = dpo_response_logps(
        policy, batch["chosen"], POLICY_DEVICE, True
    )

    policy_rejected = dpo_response_logps(
        policy, batch["rejected"], POLICY_DEVICE, True
    )

    reference_chosen = dpo_response_logps(
        reference, batch["chosen"],
        REFERENCE_DEVICE, False
    ).to(POLICY_DEVICE)

    reference_rejected = dpo_response_logps(
        reference, batch["rejected"],
        REFERENCE_DEVICE, False
    ).to(POLICY_DEVICE)

    chosen_reward = beta * (
        policy_chosen - reference_chosen
    )

    rejected_reward = beta * (
        policy_rejected - reference_rejected
    )

    margin = chosen_reward - rejected_reward

    loss = -F.logsigmoid(margin).mean()

    accuracy = (
        chosen_reward > rejected_reward
    ).float().mean()

    return {
        "loss": loss,
        "accuracy": accuracy,
        "margin": margin.mean()
    }


print("Manual DPO objective initialized.")

In [ ]:
def load_dpo_models():
    gc.collect()
    torch.cuda.empty_cache()

    policy = AutoModelForCausalLM.from_pretrained(
        str(SFT_CHECKPOINT),
        torch_dtype=torch.float32
    ).to(POLICY_DEVICE)

    reference = AutoModelForCausalLM.from_pretrained(
        str(SFT_CHECKPOINT),
        torch_dtype=torch.float16
    ).to(REFERENCE_DEVICE)

    policy.config.use_cache = False
    policy.gradient_checkpointing_enable()
    policy.train()

    reference.config.use_cache = False
    reference.eval()
    reference.requires_grad_(False)

    assert all(
        not p.requires_grad
        for p in reference.parameters()
    )

    return policy, reference


print("Model loading function ready.")

In [ ]:
@torch.no_grad()
def evaluate_dpo(policy, reference, val_loader):
    policy.eval()
    reference.eval()

    total_loss = 0.0
    total_accuracy = 0.0
    total_margin = 0.0
    total_examples = 0

    for batch in val_loader:
        metrics = dpo_batch_metrics(
            policy, reference, batch, DPO_BETA
        )

        count = batch["chosen"]["input_ids"].size(0)

        total_loss += metrics["loss"].item() * count
        total_accuracy += metrics["accuracy"].item() * count
        total_margin += metrics["margin"].item() * count
        total_examples += count

    policy.train()

    if total_examples == 0:
        raise RuntimeError("No validation examples.")

    return {
        "val_loss": total_loss / total_examples,
        "reward_accuracy": total_accuracy / total_examples,
        "reward_margin": total_margin / total_examples
    }


def save_dpo_checkpoint(policy, output_dir, name):
    path = output_dir / name
    path.mkdir(parents=True, exist_ok=True)

    policy.save_pretrained(
        path,
        safe_serialization=True
    )

    dpo_tokenizer.save_pretrained(path)

    print("Checkpoint saved:", path)


def append_dpo_metric(output_dir, record):
    with open(
        output_dir / "metrics.jsonl",
        "a",
        encoding="utf-8"
    ) as f:
        f.write(json.dumps(record) + "\n")


print("DPO validation and checkpoint functions ready.")

In [ ]:
def train_dpo_experiment(
    name,
    train_examples,
    val_examples,
    smoke=False
):
    output_dir = DPO_ROOT / (
        f"{name}_smoke" if smoke else name
    )

    output_dir.mkdir(parents=True, exist_ok=True)

    metrics_path = output_dir / "metrics.jsonl"

    if metrics_path.exists():
        raise RuntimeError(
            f"Existing metrics found in {output_dir}. "
            "Use a new run directory rather than "
            "overwriting existing results."
        )

    if smoke:
        train_examples = train_examples[:4]
        val_examples = val_examples[:4]

    train_dataset = DPODataset(train_examples)
    val_dataset = DPODataset(val_examples)

    generator = torch.Generator()
    generator.manual_seed(DPO_SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=DPO_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        collate_fn=dpo_collate
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=DPO_BATCH_SIZE,
        shuffle=False,
        collate_fn=dpo_collate
    )

    print("\nStarting:", name)
    print("Training pairs:", len(train_dataset))
    print("Validation pairs:", len(val_dataset))

    # Every experiment starts from the original SFT best.
    policy, reference = load_dpo_models()

    optimizer = torch.optim.AdamW(
        policy.parameters(),
        lr=DPO_LR
    )

    scaler = torch.amp.GradScaler("cuda")

    total_windows = (
        math.ceil(len(train_loader) / DPO_GRAD_ACCUM)
        * DPO_EPOCHS
    )

    completed_steps = 0
    skipped_updates = 0
    history = []

    best_accuracy = -1.0
    best_val_loss = float("inf")

    start_time = time.time()

    optimizer.zero_grad(set_to_none=True)

    config = {
        "experiment": name,
        "smoke": smoke,
        "initial_policy": str(SFT_CHECKPOINT),
        "reference_model": str(SFT_CHECKPOINT),
        "train_pairs": len(train_dataset),
        "validation_pairs": len(val_dataset),
        "beta": DPO_BETA,
        "learning_rate": DPO_LR,
        "optimizer": "AdamW",
        "batch_size": DPO_BATCH_SIZE,
        "gradient_accumulation": DPO_GRAD_ACCUM,
        "epochs": DPO_EPOCHS,
        "max_length": DPO_MAX_LENGTH,
        "max_prompt_length": DPO_MAX_PROMPT_LENGTH,
        "seed": DPO_SEED,
        "precision": "FP32 policy, FP16 reference",
        "prompt_template": DPO_PROMPT_TEMPLATE
    }

    with open(output_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    for epoch in range(DPO_EPOCHS):
        policy.train()

        progress = tqdm(
            train_loader,
            desc=f"{name} Epoch {epoch + 1}"
        )

        interval_losses = []

        for batch_idx, batch in enumerate(progress):

            window_start = (
                batch_idx // DPO_GRAD_ACCUM
            ) * DPO_GRAD_ACCUM

            window_end = min(
                window_start + DPO_GRAD_ACCUM,
                len(train_loader)
            )

            window_size = window_end - window_start

            metrics = dpo_batch_metrics(
                policy, reference, batch, DPO_BETA
            )

            loss = metrics["loss"]

            if not torch.isfinite(loss).item():
                raise RuntimeError(
                    f"Non-finite DPO loss at batch {batch_idx}"
                )

            interval_losses.append(loss.item())

            scaler.scale(
                loss / window_size
            ).backward()

            if batch_idx + 1 == window_end:
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    policy.parameters(),
                    DPO_GRAD_CLIP
                )

                old_scale = scaler.get_scale()

                scaler.step(optimizer)
                scaler.update()

                updated = (
                    scaler.get_scale() >= old_scale
                )

                if updated:
                    completed_steps += 1
                else:
                    skipped_updates += 1

                optimizer.zero_grad(set_to_none=True)

                progress.set_postfix(
                    loss=f"{loss.item():.4f}",
                    step=completed_steps
                )

                should_evaluate = (
                    smoke
                    or (batch_idx + 1 == len(train_loader))
                    or (
                        updated
                        and completed_steps % DPO_EVAL_EVERY == 0
                    )
                )

                if should_evaluate:
                    validation = evaluate_dpo(
                        policy,
                        reference,
                        val_loader
                    )

                    record = {
                        "step": completed_steps,
                        "epoch": epoch + 1,
                        "train_loss": (
                            sum(interval_losses)
                            / len(interval_losses)
                        ),
                        **validation,
                        "elapsed_minutes": (
                            time.time() - start_time
                        ) / 60
                    }

                    history.append(record)

                    append_dpo_metric(
                        output_dir, record
                    )

                    interval_losses.clear()

                    print(
                        f"\n{name} | Step {completed_steps}"
                        f" | DPO loss: {record['train_loss']:.4f}"
                        f" | Val loss: {validation['val_loss']:.4f}"
                        f" | Accuracy: {validation['reward_accuracy']:.4f}"
                        f" | Margin: {validation['reward_margin']:.4f}"
                    )

                    accuracy = validation["reward_accuracy"]
                    val_loss = validation["val_loss"]

                    improved = (
                        accuracy > best_accuracy
                        or (
                            accuracy == best_accuracy
                            and val_loss < best_val_loss
                        )
                    )

                    if improved:
                        best_accuracy = accuracy
                        best_val_loss = val_loss

                        save_dpo_checkpoint(
                            policy, output_dir, "best"
                        )

    save_dpo_checkpoint(
        policy, output_dir, "final"
    )

    summary = {
        "experiment": name,
        "smoke": smoke,
        "train_pairs": len(train_dataset),
        "validation_pairs": len(val_dataset),
        "planned_optimizer_windows": total_windows,
        "completed_optimizer_steps": completed_steps,
        "skipped_updates": skipped_updates,
        "best_reward_accuracy": best_accuracy,
        "best_validation_loss": best_val_loss,
        "elapsed_minutes": (
            time.time() - start_time
        ) / 60
    }

    with open(output_dir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    # Release resources before the next experiment.
    del policy
    del reference
    del optimizer
    del scaler

    gc.collect()
    torch.cuda.empty_cache()

    print("\nExperiment completed:", name)
    print(json.dumps(summary, indent=2))

    return summary

In [ ]:
def dpo_response_logps(model, batch, device, trainable):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    with torch.set_grad_enabled(trainable):
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False
        )

        logits = outputs.logits[:, :-1, :].float()
        targets = labels[:, 1:]

        valid = targets != -100
        safe_targets = targets.masked_fill(~valid, 0)

        log_probs = torch.nn.functional.log_softmax(
            logits, dim=-1
        )

        selected = log_probs.gather(
            -1, safe_targets.unsqueeze(-1)
        ).squeeze(-1)

        selected = selected.masked_fill(~valid, 0.0)

        return selected.sum(dim=-1)


print("DPO FP32 computation enabled.")

In [ ]:
scaler = torch.amp.GradScaler("cuda")

In [ ]:
# Diagnose checkpoint saving failure without rerunning training

import shutil
import os
from pathlib import Path

checkpoint_dir = (
    Path("/kaggle/working/small-model-alignment-starter")
    / "outputs/dpo/hh_rlhf_smoke_fp32/best"
)

disk = shutil.disk_usage("/kaggle/working")

print("DISK STATUS")
print("Total GB:", round(disk.total / 1024**3, 2))
print("Used GB:", round(disk.used / 1024**3, 2))
print("Free GB:", round(disk.free / 1024**3, 2))

print("\nCHECKPOINT DIRECTORY")
print("Directory exists:", checkpoint_dir.exists())

if checkpoint_dir.exists():
    for file in checkpoint_dir.iterdir():
        print(
            file.name,
            round(file.stat().st_size / 1024**2, 2),
            "MB"
        )

print("\nCHECKPOINT STORAGE TEST")

test_file = checkpoint_dir.parent / "write_test.txt"

try:
    test_file.write_text("checkpoint write test")
    print("File writing: SUCCESS")
    test_file.unlink()
except Exception as e:
    print("File writing: FAILED")
    print(type(e).__name__, str(e))

In [ ]:
import os
from pathlib import Path

paths = [
    "/kaggle/working/pa2_sft_full.zip",
    "/kaggle/working/pa2_sft_smoke.zip",
    "/kaggle/working/pa2_baseline_outputs.zip",
    "/kaggle/working/small-model-alignment-starter/outputs",
]

for path in paths:
    p = Path(path)

    if not p.exists():
        continue

    if p.is_file():
        size = p.stat().st_size
    else:
        size = sum(
            f.stat().st_size
            for f in p.rglob("*")
            if f.is_file()
        )

    print(f"{size / 1024**3:.2f} GB — {path}")

In [ ]:
from pathlib import Path
import shutil

root = Path("/kaggle/working")
project = root / "small-model-alignment-starter"

# Ensure the essential SFT checkpoint remains intact.
assert (project / "outputs/sft_full/best/config.json").exists()

# Delete downloaded duplicate ZIPs from Kaggle.
for filename in [
    "pa2_sft_smoke.zip",
    "pa2_sft_full.zip",
    "pa2_baseline_outputs.zip",
]:
    path = root / filename
    if path.exists():
        path.unlink()
        print("Deleted:", filename)

usage = shutil.disk_usage(root)
print("\nFree disk space:", round(usage.free / 1024**3, 2), "GB")

In [ ]:
import shutil
import subprocess

print("Disk usage:")
subprocess.run(["df", "-h", "/kaggle/working"])

print("\nLargest files and folders:")
subprocess.run(
    ["du", "-h", "--max-depth=3",
     "/kaggle/working/small-model-alignment-starter/outputs"]
)

print("\nMemory usage:")
subprocess.run(["free", "-h"])

In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
import shutil

root = Path("/kaggle/working")

zip_files = sorted(root.rglob("*.zip"))

print(f"Found {len(zip_files)} ZIP files:\n")

for file in zip_files:
    size_gb = file.stat().st_size / (1024**3)
    print(f"{file.name} — {size_gb:.2f} GB")
    display(FileLink(str(file)))

print("\nDownload all ZIP files using the links above.")

In [ ]:
from pathlib import Path
import shutil
import torch

root = Path("/kaggle/working")
project = root / "small-model-alignment-starter"

print("Repository:", project.exists())
print("SFT checkpoint:", (
    project / "outputs/sft_full/best/config.json"
).exists())
print("Available GPUs:", torch.cuda.device_count())
print("Free disk GB:", round(
    shutil.disk_usage(root).free / 1024**3, 2
))

In [ ]:
from pathlib import Path
import os

repo = Path("/kaggle/working/small-model-alignment-starter")

if not repo.exists():
    !git clone https://github.com/Usman-LUMS/small-model-alignment-starter.git /kaggle/working/small-model-alignment-starter
else:
    print("Repository already exists.")

os.chdir(repo)

print("\nCurrent directory:", os.getcwd())

!git log -1 --oneline
!ls

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith((".pt", ".pth", ".bin", ".safetensors", ".zip")):
            print(os.path.join(root, file))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path) / (1024**3)
        print(f"{path} | {size:.3f} GB")

In [ ]:
import os

checkpoint_path = "/kaggle/input/pa2-sft-checkpoint"

for root, dirs, files in os.walk(checkpoint_path):
    print("FOLDER:", root)
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path) / (1024**2)
        print(f"  {file} ({size:.1f} MB)")

In [ ]:
import os

print("Available inputs:")
print(os.listdir("/kaggle/input"))

print("\nCheckpoint dataset exists:",
      os.path.exists("/kaggle/input/pa2-sft-checkpoint"))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/datasets"):
    print("FOLDER:", root)
    for file in files:
        path = os.path.join(root, file)
        size = os.path.getsize(path) / (1024**2)
        print(f"  {file} ({size:.1f} MB)")

In [ ]:
from pathlib import Path
import json

SFT_PATH = Path(
    "/kaggle/input/datasets/usmanshahid890909/pa2-sft-checkpoint/best"
)

required = [
    "model.safetensors",
    "config.json",
    "tokenizer.json",
]

for name in required:
    path = SFT_PATH / name
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")

if all((SFT_PATH / name).is_file() for name in required):
    config = json.loads((SFT_PATH / "config.json").read_text())
    print("\nCheckpoint directory:", SFT_PATH)
    print("Model type:", config.get("model_type"))
    print("Checkpoint verification: PASSED")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

print("CUDA available:", torch.cuda.is_available())

tokenizer = AutoTokenizer.from_pretrained(
    str(SFT_PATH),
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    str(SFT_PATH),
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    local_files_only=True
)

print("SFT model loaded successfully!")
print("Model device:", model.device)

In [ ]:
import torch

model.eval()

prompt = "Explain the difference between supervised fine-tuning and preference optimization."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("Model response:\n")
print(response)

In [ ]:
import os

repo = "/kaggle/working/small-model-alignment-starter"

for folder in ["data", "eval_sets"]:
    path = os.path.join(repo, folder)
    print(f"\n{folder}:")
    if os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            for file in files:
                print(os.path.relpath(os.path.join(root, file), repo))
    else:
        print("Folder not found")

In [ ]:
import json
import os

data_path = os.path.join(
    repo,
    "data/persona_lora/ramsay_style_preferences.json"
)

with open(data_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Data type:", type(data).__name__)

if isinstance(data, list):
    print("Total records:", len(data))
    if data:
        print("\nFirst record:")
        print(json.dumps(data[0], indent=2)[:3000])

elif isinstance(data, dict):
    print("Top-level keys:", list(data.keys()))
    print("\nPreview:")
    print(json.dumps(data, indent=2)[:3000])

In [ ]:
from pathlib import Path

readme = Path(repo) / "README.md"
docs = Path(repo) / "docs"

print("=== README ===")
print(readme.read_text(encoding="utf-8")[:10000])

print("\n=== DOCUMENTATION FILES ===")
for path in docs.rglob("*"):
    if path.is_file():
        print(path.relative_to(repo))

In [ ]:
from datasets import load_dataset

hh = load_dataset(
    "Anthropic/hh-rlhf",
    split="train"
)

print("Total records:", len(hh))
print("Columns:", hh.column_names)
print("First example:", hh[0])

In [ ]:
import re

def split_hh_pair(example):
    chosen = example["chosen"]
    rejected = example["rejected"]

    # Identify the final assistant response
    marker = "\n\nAssistant:"
    c_pos = chosen.rfind(marker)
    r_pos = rejected.rfind(marker)

    if c_pos == -1 or r_pos == -1:
        return None

    prompt_c = chosen[:c_pos + len(marker)]
    prompt_r = rejected[:r_pos + len(marker)]

    # Both responses must share the same prompt
    if prompt_c != prompt_r:
        return None

    return {
        "prompt": prompt_c,
        "chosen": chosen[c_pos + len(marker):].strip(),
        "rejected": rejected[r_pos + len(marker):].strip()
    }

pairs = [split_hh_pair(row) for row in hh]
pairs = [
    p for p in pairs
    if p is not None and p["chosen"] and p["rejected"]
]

print("Valid DPO pairs:", len(pairs))
print("\nPrompt:", pairs[0]["prompt"][:300])
print("\nChosen:", pairs[0]["chosen"][:300])
print("\nRejected:", pairs[0]["rejected"][:300])

In [ ]:
print("Total valid pairs:", len(pairs))

print("\nChosen/rejected identical:",
      sum(p["chosen"] == p["rejected"] for p in pairs))

print("\nAverage prompt length (characters):",
      round(sum(len(p["prompt"]) for p in pairs) / len(pairs)))

print("\nAverage chosen length (characters):",
      round(sum(len(p["chosen"]) for p in pairs) / len(pairs)))

print("\nAverage rejected length (characters):",
      round(sum(len(p["rejected"]) for p in pairs) / len(pairs)))

In [ ]:
import json
import os
import random

# Remove identical preference pairs
clean_pairs = [
    p for p in pairs
    if p["chosen"].strip() != p["rejected"].strip()
]

# Shuffle reproducibly
rng = random.Random(42)
rng.shuffle(clean_pairs)

# Save the cleaned dataset
output_path = "/kaggle/working/hh_rlhf_dpo_clean.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(clean_pairs, f, ensure_ascii=False)

print("Original valid pairs:", len(pairs))
print("Cleaned DPO pairs:", len(clean_pairs))
print("Removed:", len(pairs) - len(clean_pairs))
print("Saved to:", output_path)

In [ ]:

import os
import gc
import random
import numpy as np
import torch

from pathlib import Path
from transformers import AutoTokenizer

# Release earlier inference model from GPU
if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

# Paths
PROJECT_DIR = Path(
    "/kaggle/working/small-model-alignment-starter"
)

SFT_CHECKPOINT = Path(
    "/kaggle/input/datasets/usmanshahid890909/pa2-sft-checkpoint/best"
)

DPO_OUTPUT = PROJECT_DIR / "outputs/dpo"
DPO_OUTPUT.mkdir(parents=True, exist_ok=True)

# Experiment settings (provisional)
DPO_SEED = 42
DPO_TRAIN_SIZE = 15000
DPO_VAL_SIZE = 1000

DPO_MAX_LENGTH = 512
DPO_MAX_PROMPT_LENGTH = 256

DPO_BATCH_SIZE = 1
DPO_GRAD_ACCUM = 32
DPO_EPOCHS = 1

DPO_LR = 1e-6
DPO_BETA = 0.1
DPO_GRAD_CLIP = 1.0

# GPU configuration
assert torch.cuda.device_count() >= 2, \
    "Select GPU T4 x2 in Kaggle settings."

POLICY_DEVICE = torch.device("cuda:0")
REFERENCE_DEVICE = torch.device("cuda:1")

# Reproducibility
random.seed(DPO_SEED)
np.random.seed(DPO_SEED)
torch.manual_seed(DPO_SEED)
torch.cuda.manual_seed_all(DPO_SEED)

# Load tokenizer
assert SFT_CHECKPOINT.is_dir(), "SFT checkpoint missing"

dpo_tokenizer = AutoTokenizer.from_pretrained(
    str(SFT_CHECKPOINT),
    local_files_only=True
)

if dpo_tokenizer.pad_token_id is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

DPO_PROMPT_TEMPLATE = (
    "### Instruction:\n{instruction}\n\n"
    "### Response:\n"
)

print("DPO configuration initialized!")
print("Checkpoint:", SFT_CHECKPOINT)
print("GPUs:", torch.cuda.device_count())
print("Training pairs:", DPO_TRAIN_SIZE)
print("Validation pairs:", DPO_VAL_SIZE)
print("DPO beta:", DPO_BETA)


In [ ]:

import json
from pathlib import Path

# Load previously cleaned HH-RLHF preference pairs
CLEAN_DATA_PATH = Path(
    "/kaggle/working/hh_rlhf_dpo_clean.json"
)

assert CLEAN_DATA_PATH.is_file(), "Cleaned dataset not found"

with open(CLEAN_DATA_PATH, "r", encoding="utf-8") as f:
    clean_pairs = json.load(f)

# Deduplicate complete preference records
seen = set()
unique_pairs = []

for pair in clean_pairs:
    key = (
        pair["prompt"],
        pair["chosen"],
        pair["rejected"]
    )
    if key not in seen:
        seen.add(key)
        unique_pairs.append(pair)

# Reproducible shuffle
rng = random.Random(DPO_SEED)
rng.shuffle(unique_pairs)

assert len(unique_pairs) >= DPO_TRAIN_SIZE + DPO_VAL_SIZE

# Separate training and validation
hh_train_pairs = unique_pairs[:DPO_TRAIN_SIZE]

hh_val_pairs = unique_pairs[
    DPO_TRAIN_SIZE:DPO_TRAIN_SIZE + DPO_VAL_SIZE
]

# Verify no overlap
train_keys = {
    (p["prompt"], p["chosen"], p["rejected"])
    for p in hh_train_pairs
}

val_keys = {
    (p["prompt"], p["chosen"], p["rejected"])
    for p in hh_val_pairs
}

assert train_keys.isdisjoint(val_keys)

print("HH-RLHF dataset prepared!")
print("Available unique pairs:", len(unique_pairs))
print("Training pairs:", len(hh_train_pairs))
print("Validation pairs:", len(hh_val_pairs))
print("Train/validation overlap:", len(train_keys & val_keys))
print("\nExample prompt:")
print(hh_train_pairs[0]["prompt"][:200])


In [ ]:

import torch

IGNORE_INDEX = -100

def tokenize_dpo_pair(pair):
    prompt = pair["prompt"]
    chosen = pair["chosen"]
    rejected = pair["rejected"]

    # Reserve the ending of the prompt to preserve
    # the most recent conversation and Assistant marker.
    prompt_ids = dpo_tokenizer(
        prompt, add_special_tokens=False
    )["input_ids"][-DPO_MAX_PROMPT_LENGTH:]

    eos_id = dpo_tokenizer.eos_token_id

    def encode_response(response):
        response_ids = dpo_tokenizer(
            response, add_special_tokens=False
        )["input_ids"]

        max_response = DPO_MAX_LENGTH - len(prompt_ids)
        if eos_id is not None and max_response > 0:
            response_ids = response_ids[:max_response - 1] + [eos_id]
        else:
            response_ids = response_ids[:max_response]

        input_ids = prompt_ids + response_ids

        # Mask prompt: calculate log probabilities only
        # for assistant response tokens.
        labels = (
            [IGNORE_INDEX] * len(prompt_ids)
            + response_ids
        )

        return {
            "input_ids": input_ids,
            "attention_mask": [1] * len(input_ids),
            "labels": labels,
        }

    return {
        "chosen": encode_response(chosen),
        "rejected": encode_response(rejected),
    }


# Test tokenization on one preference pair
sample = tokenize_dpo_pair(hh_train_pairs[0])

for name in ["chosen", "rejected"]:
    item = sample[name]
    prompt_masked = sum(x == IGNORE_INDEX for x in item["labels"])
    response_tokens = sum(x != IGNORE_INDEX for x in item["labels"])

    assert len(item["input_ids"]) == len(item["labels"])
    assert len(item["input_ids"]) <= DPO_MAX_LENGTH
    assert response_tokens > 0

    print(f"\n{name.upper()}")
    print("Total tokens:", len(item["input_ids"]))
    print("Masked prompt tokens:", prompt_masked)
    print("Response tokens:", response_tokens)

print("\nCell 3 tokenization verification: PASSED")


In [ ]:

import torch
import torch.nn.functional as F

def compute_sequence_logps(model, batch):
    """
    Compute summed log probabilities over response
    tokens only, excluding prompt and padding.
    """
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"]
    )

    logits = outputs.logits[:, :-1, :].float()
    labels = batch["labels"][:, 1:]

    mask = labels != IGNORE_INDEX
    safe_labels = labels.masked_fill(~mask, 0)

    log_probs = F.log_softmax(logits, dim=-1)

    token_logps = torch.gather(
        log_probs,
        dim=-1,
        index=safe_labels.unsqueeze(-1)
    ).squeeze(-1)

    return (token_logps * mask).sum(dim=-1)


def compute_dpo_loss(
    policy_chosen_logps,
    policy_rejected_logps,
    ref_chosen_logps,
    ref_rejected_logps,
    beta=DPO_BETA
):
    """
    Standard DPO loss using reference-adjusted
    chosen/rejected log-probability differences.
    """
    policy_logratios = (
        policy_chosen_logps - policy_rejected_logps
    )

    reference_logratios = (
        ref_chosen_logps - ref_rejected_logps
    )

    logits = beta * (
        policy_logratios - reference_logratios
    )

    losses = -F.logsigmoid(logits)

    rewards_chosen = beta * (
        policy_chosen_logps - ref_chosen_logps
    )

    rewards_rejected = beta * (
        policy_rejected_logps - ref_rejected_logps
    )

    accuracy = (
        rewards_chosen > rewards_rejected
    ).float().mean()

    return {
        "loss": losses.mean(),
        "preference_accuracy": accuracy,
        "reward_margin": (
            rewards_chosen - rewards_rejected
        ).mean()
    }


# Test loss implementation
pc = torch.tensor([-10.0, -8.0], requires_grad=True)
pr = torch.tensor([-12.0, -11.0])
rc = torch.tensor([-10.5, -8.5])
rr = torch.tensor([-11.5, -10.5])

result = compute_dpo_loss(pc, pr, rc, rr)

assert torch.isfinite(result["loss"])
result["loss"].backward()
assert pc.grad is not None
assert torch.isfinite(pc.grad).all()

print("DPO loss:", result["loss"].item())
print("Preference accuracy:",
      result["preference_accuracy"].item())
print("Reward margin:",
      result["reward_margin"].item())
print("Gradient verified:", pc.grad)
print("\nCell 4 DPO loss verification: PASSED")


In [ ]:

import gc
import torch
from transformers import AutoModelForCausalLM

gc.collect()
torch.cuda.empty_cache()

# Trainable policy model on GPU 0
policy_model = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    dtype=torch.float32,
    local_files_only=True,
    low_cpu_mem_usage=True
).to(POLICY_DEVICE)

# Frozen reference model on GPU 1
reference_model = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    dtype=torch.float16,
    local_files_only=True,
    low_cpu_mem_usage=True
).to(REFERENCE_DEVICE)

# Enable training only for policy
policy_model.train()
policy_model.config.use_cache = False

# Freeze reference parameters
reference_model.eval()
reference_model.requires_grad_(False)

# Gradient checkpointing reduces memory use
policy_model.gradient_checkpointing_enable()

# Verify devices and trainability
assert next(policy_model.parameters()).device.type == "cuda"
assert next(reference_model.parameters()).device.type == "cuda"

assert next(policy_model.parameters()).device.index == 0
assert next(reference_model.parameters()).device.index == 1

assert any(p.requires_grad for p in policy_model.parameters())
assert not any(p.requires_grad for p in reference_model.parameters())

print("Policy model:", next(policy_model.parameters()).device)
print("Reference model:", next(reference_model.parameters()).device)

print("Policy trainable parameters:",
      sum(p.numel() for p in policy_model.parameters()
          if p.requires_grad))

print("Reference trainable parameters:",
      sum(p.numel() for p in reference_model.parameters()
          if p.requires_grad))

print("\nCell 5: PASSED")


In [ ]:

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class DPODataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return tokenize_dpo_pair(self.pairs[idx])


def dpo_collate_fn(batch):
    result = {}

    for preference in ["chosen", "rejected"]:
        examples = [item[preference] for item in batch]

        input_ids = [
            torch.tensor(e["input_ids"], dtype=torch.long)
            for e in examples
        ]

        labels = [
            torch.tensor(e["labels"], dtype=torch.long)
            for e in examples
        ]

        attention_masks = [
            torch.tensor(e["attention_mask"], dtype=torch.long)
            for e in examples
        ]

        result[preference] = {
            "input_ids": pad_sequence(
                input_ids,
                batch_first=True,
                padding_value=dpo_tokenizer.pad_token_id
            ),
            "attention_mask": pad_sequence(
                attention_masks,
                batch_first=True,
                padding_value=0
            ),
            "labels": pad_sequence(
                labels,
                batch_first=True,
                padding_value=IGNORE_INDEX
            )
        }

    return result


# Create datasets
hh_train_dataset = DPODataset(hh_train_pairs)
hh_val_dataset = DPODataset(hh_val_pairs)

# Create dataloaders
hh_train_loader = DataLoader(
    hh_train_dataset,
    batch_size=DPO_BATCH_SIZE,
    shuffle=True,
    collate_fn=dpo_collate_fn,
    num_workers=0
)

hh_val_loader = DataLoader(
    hh_val_dataset,
    batch_size=DPO_BATCH_SIZE,
    shuffle=False,
    collate_fn=dpo_collate_fn,
    num_workers=0
)

# Verify a sample batch
batch = next(iter(hh_train_loader))

for preference in ["chosen", "rejected"]:
    data = batch[preference]

    assert data["input_ids"].shape == data["labels"].shape
    assert data["attention_mask"].shape == data["input_ids"].shape
    assert (data["labels"] != IGNORE_INDEX).sum() > 0

    print(
        preference,
        "input shape:",
        tuple(data["input_ids"].shape)
    )

print("\nTraining batches:", len(hh_train_loader))
print("Validation batches:", len(hh_val_loader))
print("Cell 6: PASSED")


In [ ]:

import torch
import torch.nn.functional as F
from torch.optim import AdamW

# Four examples, one optimizer update
smoke_pairs = hh_train_pairs[:4]

smoke_loader = DataLoader(
    DPODataset(smoke_pairs),
    batch_size=1,
    shuffle=False,
    collate_fn=dpo_collate_fn
)

optimizer = AdamW(
    policy_model.parameters(),
    lr=DPO_LR
)

policy_model.train()
reference_model.eval()

optimizer.zero_grad(set_to_none=True)

smoke_losses = []
smoke_accuracies = []

for step, batch in enumerate(smoke_loader):

    # Reference forward pass on GPU 1
    with torch.no_grad():
        ref_logps = {}

        for preference in ["chosen", "rejected"]:
            ref_batch = {
                k: v.to(REFERENCE_DEVICE)
                for k, v in batch[preference].items()
            }

            ref_logps[preference] = compute_sequence_logps(
                reference_model,
                ref_batch
            ).to(POLICY_DEVICE)

    # Policy forward pass on GPU 0
    policy_logps = {}

    for preference in ["chosen", "rejected"]:
        policy_batch = {
            k: v.to(POLICY_DEVICE)
            for k, v in batch[preference].items()
        }

        policy_logps[preference] = compute_sequence_logps(
            policy_model,
            policy_batch
        )

    metrics = compute_dpo_loss(
        policy_logps["chosen"],
        policy_logps["rejected"],
        ref_logps["chosen"],
        ref_logps["rejected"]
    )

    assert torch.isfinite(metrics["loss"]), "Non-finite loss"

    # Accumulate gradients across four pairs
    (metrics["loss"] / len(smoke_loader)).backward()

    smoke_losses.append(metrics["loss"].item())
    smoke_accuracies.append(
        metrics["preference_accuracy"].item()
    )

    print(
        f"Pair {step + 1}/4 | "
        f"Loss: {smoke_losses[-1]:.4f} | "
        f"Accuracy: {smoke_accuracies[-1]:.2f}"
    )

# Validate gradients
grads = [
    p.grad for p in policy_model.parameters()
    if p.grad is not None
]

assert grads, "No gradients produced"
assert all(torch.isfinite(g).all() for g in grads), \
    "Non-finite gradients"

grad_norm = torch.nn.utils.clip_grad_norm_(
    policy_model.parameters(),
    DPO_GRAD_CLIP
)

assert torch.isfinite(grad_norm), "Invalid gradient norm"

optimizer.step()
optimizer.zero_grad(set_to_none=True)

print("\nSmoke test completed")
print("Average DPO loss:", sum(smoke_losses) / len(smoke_losses))
print("Average preference accuracy:",
      sum(smoke_accuracies) / len(smoke_accuracies))
print("Gradient norm:", grad_norm.item())
print("Optimizer updates: 1")
print("Cell 7: PASSED")


In [ ]:

import gc
import torch
from transformers import AutoModelForCausalLM
from torch.optim import AdamW

# Release smoke-test policy and optimizer
del optimizer
del policy_model

gc.collect()
torch.cuda.empty_cache()

# Reload original SFT weights
policy_model = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    dtype=torch.float32,
    local_files_only=True,
    low_cpu_mem_usage=True
).to(POLICY_DEVICE)

policy_model.config.use_cache = False
policy_model.gradient_checkpointing_enable()
policy_model.train()

# Reference model remains frozen on GPU 1
reference_model.eval()
reference_model.requires_grad_(False)

# Fresh optimizer
optimizer = AdamW(
    policy_model.parameters(),
    lr=DPO_LR,
    weight_decay=0.0
)

optimizer.zero_grad(set_to_none=True)

# Training output directory
HH_OUTPUT_DIR = DPO_OUTPUT / "hh_rlhf"
HH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training configuration
import math

total_batches = len(hh_train_loader)
total_updates = math.ceil(
    total_batches / DPO_GRAD_ACCUM
) * DPO_EPOCHS

print("Original SFT policy restored")
print("Policy device:", next(policy_model.parameters()).device)
print("Reference device:", next(reference_model.parameters()).device)
print("Training pairs:", len(hh_train_dataset))
print("Epochs:", DPO_EPOCHS)
print("Batch size:", DPO_BATCH_SIZE)
print("Gradient accumulation:", DPO_GRAD_ACCUM)
print("Expected optimizer updates:", total_updates)
print("Output directory:", HH_OUTPUT_DIR)

print("\nCell 8: PASSED")


In [ ]:

import json
import math
import time
import torch
from tqdm.auto import tqdm

# Training history
hh_training_history = []

policy_model.train()
reference_model.eval()

optimizer.zero_grad(set_to_none=True)

global_step = 0
optimizer_updates = 0
accum_count = 0
accum_loss = 0.0
accum_accuracy = 0.0

start_time = time.time()

for epoch in range(DPO_EPOCHS):

    progress = tqdm(
        hh_train_loader,
        desc=f"HH-RLHF DPO Epoch {epoch + 1}"
    )

    for batch_idx, batch in enumerate(progress):

        # 1. Reference model: GPU 1
        with torch.no_grad():
            ref_logps = {}

            for preference in ["chosen", "rejected"]:
                ref_batch = {
                    k: v.to(REFERENCE_DEVICE)
                    for k, v in batch[preference].items()
                }

                ref_logps[preference] = compute_sequence_logps(
                    reference_model,
                    ref_batch
                ).to(POLICY_DEVICE)

        # 2. Trainable policy: GPU 0
        policy_logps = {}

        for preference in ["chosen", "rejected"]:
            policy_batch = {
                k: v.to(POLICY_DEVICE)
                for k, v in batch[preference].items()
            }

            policy_logps[preference] = compute_sequence_logps(
                policy_model,
                policy_batch
            )

        # 3. Compute DPO objective
        metrics = compute_dpo_loss(
            policy_logps["chosen"],
            policy_logps["rejected"],
            ref_logps["chosen"],
            ref_logps["rejected"],
            beta=DPO_BETA
        )

        loss = metrics["loss"]

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite DPO loss at batch {batch_idx}"
            )

        # 4. Gradient accumulation
        loss.backward()

        accum_count += 1
        accum_loss += loss.item()
        accum_accuracy += metrics[
            "preference_accuracy"
        ].item()

        global_step += 1

        is_last_batch = (
            batch_idx + 1 == len(hh_train_loader)
        )

        if (
            accum_count == DPO_GRAD_ACCUM
            or is_last_batch
        ):

            # Average accumulated gradients
            for param in policy_model.parameters():
                if param.grad is not None:
                    param.grad.div_(accum_count)

            grad_norm = torch.nn.utils.clip_grad_norm_(
                policy_model.parameters(),
                DPO_GRAD_CLIP
            )

            if not torch.isfinite(grad_norm):
                raise RuntimeError(
                    "Non-finite gradient norm"
                )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            optimizer_updates += 1

            avg_loss = accum_loss / accum_count
            avg_accuracy = accum_accuracy / accum_count

            record = {
                "epoch": epoch + 1,
                "update": optimizer_updates,
                "step": global_step,
                "loss": avg_loss,
                "preference_accuracy": avg_accuracy,
                "grad_norm": float(grad_norm.item())
            }

            hh_training_history.append(record)

            progress.set_postfix({
                "loss": f"{avg_loss:.4f}",
                "accuracy": f"{avg_accuracy:.3f}",
                "updates": optimizer_updates
            })

            accum_count = 0
            accum_loss = 0.0
            accum_accuracy = 0.0

    # Save progress after each epoch
    history_path = HH_OUTPUT_DIR / "training_history.json"

    with open(history_path, "w") as f:
        json.dump(hh_training_history, f, indent=2)

elapsed_minutes = (time.time() - start_time) / 60

print("\nHH-RLHF DPO training completed!")
print("Total batches:", global_step)
print("Optimizer updates:", optimizer_updates)
print("Expected updates:", total_updates)
print("Training time (minutes):", round(elapsed_minutes, 2))
print("Final DPO loss:", hh_training_history[-1]["loss"])
print("Final preference accuracy:",
      hh_training_history[-1]["preference_accuracy"])

assert optimizer_updates == total_updates

print("\nCell 9: PASSED")


In [ ]:

import json
import torch
from pathlib import Path

# Save trained policy model
HH_CHECKPOINT = HH_OUTPUT_DIR / "final"
HH_CHECKPOINT.mkdir(parents=True, exist_ok=True)

policy_model.eval()

policy_model.save_pretrained(
    str(HH_CHECKPOINT),
    safe_serialization=True
)

dpo_tokenizer.save_pretrained(
    str(HH_CHECKPOINT)
)

# Save training configuration and summary
training_summary = {
    "experiment": "HH-RLHF DPO",
    "base_checkpoint": str(SFT_CHECKPOINT),
    "training_pairs": len(hh_train_dataset),
    "validation_pairs": len(hh_val_dataset),
    "epochs": DPO_EPOCHS,
    "beta": DPO_BETA,
    "learning_rate": DPO_LR,
    "batch_size": DPO_BATCH_SIZE,
    "gradient_accumulation": DPO_GRAD_ACCUM,
    "optimizer_updates": optimizer_updates,
    "training_minutes": round(elapsed_minutes, 2),
    "final_batch_loss": hh_training_history[-1]["loss"],
    "final_batch_preference_accuracy":
        hh_training_history[-1]["preference_accuracy"]
}

with open(
    HH_OUTPUT_DIR / "training_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(training_summary, f, indent=2)

# Verify saved checkpoint
assert (HH_CHECKPOINT / "config.json").exists()
assert (HH_CHECKPOINT / "model.safetensors").exists()

print("HH-RLHF checkpoint saved successfully!")
print("Checkpoint:", HH_CHECKPOINT)
print("Files:")

for file in HH_CHECKPOINT.iterdir():
    print(" -", file.name)

print("\nCell 10: PASSED")

In [ ]:

import zipfile
import json
from pathlib import Path

archive_path = Path("/kaggle/working/pa2_dpo_hh_rlhf.zip")

files_to_save = [
    HH_OUTPUT_DIR / "final",
    HH_OUTPUT_DIR / "training_history.json",
    HH_OUTPUT_DIR / "training_summary.json",
]

with zipfile.ZipFile(
    archive_path,
    mode="w",
    compression=zipfile.ZIP_STORED
) as archive:

    for item in files_to_save:
        if item.is_dir():
            for file in item.rglob("*"):
                if file.is_file():
                    archive.write(
                        file,
                        arcname=file.relative_to(HH_OUTPUT_DIR)
                    )
        elif item.is_file():
            archive.write(
                item,
                arcname=item.name
            )

# Verify archive
with zipfile.ZipFile(archive_path) as archive:
    contents = archive.namelist()
    assert "final/model.safetensors" in contents
    assert "final/config.json" in contents
    assert "training_history.json" in contents
    assert archive.testzip() is None

print("Archive created successfully!")
print("Location:", archive_path)
print(
    "Size:",
    round(archive_path.stat().st_size / (1024**3), 2),
    "GB"
)
print("Files archived:", len(contents))
print("\nCell 11: PASSED")


In [ ]:

import json
import torch
from pathlib import Path
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM

# Paths
BASE = Path("/kaggle/working/small-model-alignment-starter")
HH_OUTPUT_DIR = BASE / "outputs/dpo/hh_rlhf"
HH_CHECKPOINT = HH_OUTPUT_DIR / "final"

SFT_CHECKPOINT = Path(
    "/kaggle/input/datasets/usmanshahid890909/pa2-sft-checkpoint/best"
)

# Check that saved model weights are still present
assert (HH_CHECKPOINT / "model.safetensors").exists(), (
    "HH-RLHF checkpoint not found. Check saved Kaggle outputs."
)

assert SFT_CHECKPOINT.exists(), "SFT checkpoint not found"

POLICY_DEVICE = torch.device("cuda:0")
REFERENCE_DEVICE = torch.device("cuda:1")
DPO_BETA = 0.1
IGNORE_INDEX = -100
DPO_MAX_LENGTH = 512
DPO_MAX_PROMPT_LENGTH = 256
DPO_BATCH_SIZE = 1

# Restore tokenizer
dpo_tokenizer = AutoTokenizer.from_pretrained(
    str(HH_CHECKPOINT),
    local_files_only=True
)

if dpo_tokenizer.pad_token_id is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

# Restore trained DPO policy
policy_model = AutoModelForCausalLM.from_pretrained(
    str(HH_CHECKPOINT),
    dtype=torch.float16,
    local_files_only=True
).to(POLICY_DEVICE)

policy_model.eval()

# Restore original frozen SFT reference
reference_model = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    dtype=torch.float16,
    local_files_only=True
).to(REFERENCE_DEVICE)

reference_model.eval()
reference_model.requires_grad_(False)

print("HH-RLHF DPO policy restored:", policy_model.device)
print("SFT reference restored:", reference_model.device)
print("Cell 12A: PASSED")


In [ ]:

import os
from pathlib import Path

print("=== Available Kaggle Inputs ===")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if (
            file.endswith(".zip")
            or file == "model.safetensors"
        ):
            path = Path(root) / file
            size = path.stat().st_size / (1024**3)

            print(f"{path} | {size:.2f} GB")

print("\n=== Kaggle Working Directory ===")

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if (
            file.endswith(".zip")
            or file == "model.safetensors"
        ):
            path = Path(root) / file
            size = path.stat().st_size / (1024**3)

            print(f"{path} | {size:.2f} GB")

print("\nSearch completed.")


In [ ]:

from pathlib import Path

# Full SFT checkpoint recovered from Kaggle notebook input
FULL_SFT = Path(
    "/kaggle/input/notebooks/usmanshahid890909/"
    "notebook105a4f89ef/small-model-alignment-starter/"
    "outputs/sft_full/best"
)

# HH-RLHF DPO checkpoint from previous training
HH_DPO = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/dpo/hh_rlhf/final"
)

print("Full SFT checkpoint:",
      (FULL_SFT / "model.safetensors").exists())

print("HH-RLHF DPO checkpoint:",
      (HH_DPO / "model.safetensors").exists())

print("\nChecking available notebook outputs:")

output_root = Path(
    "/kaggle/input/notebooks/usmanshahid890909/"
    "notebook105a4f89ef"
)

for path in output_root.rglob("*"):
    if path.is_file() and (
        "dpo" in str(path).lower()
        or path.suffix == ".zip"
    ):
        print(path)


In [ ]:

from pathlib import Path

search_roots = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path("/kaggle/temp")
]

found = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if not path.is_file():
            continue

        name = path.name.lower()
        location = str(path).lower()

        if (
            "pa2_dpo_hh_rlhf" in name
            or (
                "hh_rlhf" in location
                and name == "model.safetensors"
            )
        ):
            found.append(path)

print("Recovered DPO files:", len(found))

for path in found:
    print(path)
    print(
        "Size (GB):",
        round(path.stat().st_size / (1024**3), 2)
    )

print("\nFull SFT available:", FULL_SFT.exists())
print("Cell 12D: COMPLETE")


In [ ]:

from pathlib import Path

FULL_SFT = Path(
    "/kaggle/input/notebooks/usmanshahid890909/"
    "notebook105a4f89ef/small-model-alignment-starter/"
    "outputs/sft_full/best"
)

required = [
    "model.safetensors",
    "config.json",
    "tokenizer.json"
]

for filename in required:
    print(
        filename,
        "FOUND" if (FULL_SFT / filename).is_file()
        else "MISSING"
    )

assert all((FULL_SFT / f).is_file() for f in required)

print("\nFull SFT checkpoint ready for DPO retraining.")


In [ ]:

import os
import gc
import json
import random
import math
import torch
import numpy as np

from pathlib import Path
from transformers import AutoTokenizer

# Use recovered FULL SFT checkpoint
SFT_CHECKPOINT = FULL_SFT

PROJECT_DIR = Path(
    "/kaggle/working/small-model-alignment-starter"
)

DPO_OUTPUT = PROJECT_DIR / "outputs/dpo"
HH_OUTPUT_DIR = DPO_OUTPUT / "hh_rlhf"
HH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
DPO_SEED = 42
DPO_TRAIN_SIZE = 15000
DPO_VAL_SIZE = 1000

DPO_MAX_LENGTH = 512
DPO_MAX_PROMPT_LENGTH = 256

DPO_BATCH_SIZE = 1
DPO_GRAD_ACCUM = 32
DPO_EPOCHS = 1

DPO_LR = 1e-6
DPO_BETA = 0.1
DPO_GRAD_CLIP = 1.0
IGNORE_INDEX = -100

assert torch.cuda.device_count() >= 2, \
    "Enable GPU T4 x2 in Kaggle."

POLICY_DEVICE = torch.device("cuda:0")
REFERENCE_DEVICE = torch.device("cuda:1")

random.seed(DPO_SEED)
np.random.seed(DPO_SEED)
torch.manual_seed(DPO_SEED)
torch.cuda.manual_seed_all(DPO_SEED)

dpo_tokenizer = AutoTokenizer.from_pretrained(
    str(SFT_CHECKPOINT),
    local_files_only=True
)

if dpo_tokenizer.pad_token_id is None:
    dpo_tokenizer.pad_token = dpo_tokenizer.eos_token

print("Full SFT checkpoint:", SFT_CHECKPOINT)
print("GPUs:", torch.cuda.device_count())
print("Training pairs:", DPO_TRAIN_SIZE)
print("Validation pairs:", DPO_VAL_SIZE)
print("Beta:", DPO_BETA)
print("Configuration restored successfully!")

In [ ]:

from datasets import load_dataset
from pathlib import Path
import json
import random

# Load HH-RLHF training data
hh = load_dataset(
    "Anthropic/hh-rlhf",
    split="train"
)

def split_hh_pair(example):
    chosen = example["chosen"]
    rejected = example["rejected"]

    marker = "\n\nAssistant:"
    c_pos = chosen.rfind(marker)
    r_pos = rejected.rfind(marker)

    if c_pos == -1 or r_pos == -1:
        return None

    prompt_c = chosen[:c_pos + len(marker)]
    prompt_r = rejected[:r_pos + len(marker)]

    if prompt_c != prompt_r:
        return None

    chosen_response = chosen[c_pos + len(marker):].strip()
    rejected_response = rejected[r_pos + len(marker):].strip()

    if (
        not chosen_response
        or not rejected_response
        or chosen_response == rejected_response
    ):
        return None

    return {
        "prompt": prompt_c,
        "chosen": chosen_response,
        "rejected": rejected_response
    }

# Convert and deduplicate
seen = set()
clean_pairs = []

for row in hh:
    pair = split_hh_pair(row)

    if pair is None:
        continue

    key = (
        pair["prompt"],
        pair["chosen"],
        pair["rejected"]
    )

    if key not in seen:
        seen.add(key)
        clean_pairs.append(pair)

# Reproducible selection
rng = random.Random(DPO_SEED)
rng.shuffle(clean_pairs)

assert len(clean_pairs) >= DPO_TRAIN_SIZE + DPO_VAL_SIZE

hh_train_pairs = clean_pairs[:DPO_TRAIN_SIZE]
hh_val_pairs = clean_pairs[
    DPO_TRAIN_SIZE:DPO_TRAIN_SIZE + DPO_VAL_SIZE
]

# Save split manifest so the same samples
# can be restored after another session reset
split_path = Path("/kaggle/working/hh_dpo_split.json")

with open(split_path, "w", encoding="utf-8") as f:
    json.dump({
        "train": hh_train_pairs,
        "validation": hh_val_pairs
    }, f, ensure_ascii=False)

print("Clean preference pairs:", len(clean_pairs))
print("Training pairs:", len(hh_train_pairs))
print("Validation pairs:", len(hh_val_pairs))
print("Split saved:", split_path)
print("Dataset restoration: PASSED")


In [ ]:

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# 1. Tokenization
def tokenize_dpo_pair(pair):
    prompt_ids = dpo_tokenizer(
        pair["prompt"],
        add_special_tokens=False
    )["input_ids"][-DPO_MAX_PROMPT_LENGTH:]

    def encode_response(response):
        response_ids = dpo_tokenizer(
            response,
            add_special_tokens=False
        )["input_ids"]

        available = DPO_MAX_LENGTH - len(prompt_ids)

        if dpo_tokenizer.eos_token_id is not None:
            response_ids = (
                response_ids[:available - 1]
                + [dpo_tokenizer.eos_token_id]
            )
        else:
            response_ids = response_ids[:available]

        return {
            "input_ids": prompt_ids + response_ids,
            "attention_mask": [1] * (
                len(prompt_ids) + len(response_ids)
            ),
            "labels": (
                [IGNORE_INDEX] * len(prompt_ids)
                + response_ids
            )
        }

    return {
        "chosen": encode_response(pair["chosen"]),
        "rejected": encode_response(pair["rejected"])
    }


# 2. Dataset
class DPODataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return tokenize_dpo_pair(self.pairs[idx])


# 3. Collator
def dpo_collate_fn(batch):
    result = {}

    for preference in ["chosen", "rejected"]:
        examples = [item[preference] for item in batch]

        result[preference] = {}

        for key in ["input_ids", "attention_mask", "labels"]:
            tensors = [
                torch.tensor(e[key], dtype=torch.long)
                for e in examples
            ]

            padding = (
                IGNORE_INDEX if key == "labels"
                else 0 if key == "attention_mask"
                else dpo_tokenizer.pad_token_id
            )

            result[preference][key] = pad_sequence(
                tensors,
                batch_first=True,
                padding_value=padding
            )

    return result


# 4. Response-only log probabilities
def compute_sequence_logps(model, batch):
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"]
    )

    logits = outputs.logits[:, :-1, :].float()
    labels = batch["labels"][:, 1:]

    mask = labels != IGNORE_INDEX
    safe_labels = labels.masked_fill(~mask, 0)

    token_logps = F.log_softmax(
        logits, dim=-1
    ).gather(
        -1, safe_labels.unsqueeze(-1)
    ).squeeze(-1)

    return (token_logps * mask).sum(dim=-1)


# 5. DPO loss
def compute_dpo_loss(
    policy_chosen,
    policy_rejected,
    ref_chosen,
    ref_rejected,
    beta=DPO_BETA
):
    logits = beta * (
        (policy_chosen - policy_rejected)
        - (ref_chosen - ref_rejected)
    )

    rewards_chosen = beta * (
        policy_chosen - ref_chosen
    )

    rewards_rejected = beta * (
        policy_rejected - ref_rejected
    )

    return {
        "loss": -F.logsigmoid(logits).mean(),
        "preference_accuracy": (
            rewards_chosen > rewards_rejected
        ).float().mean(),
        "reward_margin": (
            rewards_chosen - rewards_rejected
        ).mean()
    }


# 6. DataLoaders
hh_train_loader = DataLoader(
    DPODataset(hh_train_pairs),
    batch_size=DPO_BATCH_SIZE,
    shuffle=True,
    collate_fn=dpo_collate_fn
)

hh_val_loader = DataLoader(
    DPODataset(hh_val_pairs),
    batch_size=DPO_BATCH_SIZE,
    shuffle=False,
    collate_fn=dpo_collate_fn
)

# Verification
sample = next(iter(hh_train_loader))

assert sample["chosen"]["input_ids"].shape == \
       sample["chosen"]["labels"].shape

assert (sample["chosen"]["labels"] != IGNORE_INDEX).any()
assert (sample["rejected"]["labels"] != IGNORE_INDEX).any()

print("Training batches:", len(hh_train_loader))
print("Validation batches:", len(hh_val_loader))
print("DPO functions restored successfully!")
print("Cell PASSED")


In [ ]:

import gc
import torch
from transformers import AutoModelForCausalLM
from torch.optim import AdamW

gc.collect()
torch.cuda.empty_cache()

# Load full SFT checkpoint as trainable policy
policy_model = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    dtype=torch.float32,
    local_files_only=True,
    low_cpu_mem_usage=True
).to(POLICY_DEVICE)

policy_model.config.use_cache = False
policy_model.gradient_checkpointing_enable()
policy_model.train()

# Load frozen reference model
reference_model = AutoModelForCausalLM.from_pretrained(
    str(SFT_CHECKPOINT),
    dtype=torch.float16,
    local_files_only=True,
    low_cpu_mem_usage=True
).to(REFERENCE_DEVICE)

reference_model.eval()
reference_model.requires_grad_(False)

# Fresh optimizer
optimizer = AdamW(
    policy_model.parameters(),
    lr=DPO_LR,
    weight_decay=0.0
)

optimizer.zero_grad(set_to_none=True)

# Training configuration
total_batches = len(hh_train_loader)
total_updates = math.ceil(
    total_batches / DPO_GRAD_ACCUM
) * DPO_EPOCHS

# Checkpoint frequency
SAVE_EVERY_UPDATES = 100

# Checkpoint directory
HH_OUTPUT_DIR = DPO_OUTPUT / "hh_rlhf"
HH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Policy:", next(policy_model.parameters()).device)
print("Reference:", next(reference_model.parameters()).device)
print("Expected updates:", total_updates)
print("Checkpoint interval:", SAVE_EVERY_UPDATES)
print("Checkpoint directory:", HH_OUTPUT_DIR)
print("Models restored successfully!")


In [ ]:

import json
import time
import torch
from tqdm.auto import tqdm

hh_training_history = []
optimizer_updates = 0
global_step = 0

policy_model.train()
reference_model.eval()
optimizer.zero_grad(set_to_none=True)

start_time = time.time()

def save_dpo_checkpoint(update):
    checkpoint_dir = HH_OUTPUT_DIR / f"checkpoint-{update}"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    policy_model.save_pretrained(
        str(checkpoint_dir),
        safe_serialization=True
    )
    dpo_tokenizer.save_pretrained(str(checkpoint_dir))

    # Save optimizer state for training resumption
    torch.save(
        {
            "optimizer": optimizer.state_dict(),
            "optimizer_updates": update,
            "global_step": global_step,
            "rng_state": torch.get_rng_state()
        },
        checkpoint_dir / "training_state.pt"
    )

    with open(
        HH_OUTPUT_DIR / "training_history.json",
        "w"
    ) as f:
        json.dump(hh_training_history, f, indent=2)

    print(f"\nCheckpoint saved at update {update}")


for epoch in range(DPO_EPOCHS):

    progress = tqdm(
        hh_train_loader,
        desc=f"HH-RLHF Epoch {epoch + 1}"
    )

    accum_count = 0
    accum_loss = 0.0
    accum_accuracy = 0.0

    for batch_idx, batch in enumerate(progress):

        # Frozen reference model (GPU 1)
        with torch.no_grad():
            ref_logps = {}

            for preference in ["chosen", "rejected"]:
                ref_batch = {
                    k: v.to(REFERENCE_DEVICE)
                    for k, v in batch[preference].items()
                }

                ref_logps[preference] = compute_sequence_logps(
                    reference_model,
                    ref_batch
                ).to(POLICY_DEVICE)

        # Trainable policy model (GPU 0)
        policy_logps = {}

        for preference in ["chosen", "rejected"]:
            policy_batch = {
                k: v.to(POLICY_DEVICE)
                for k, v in batch[preference].items()
            }

            policy_logps[preference] = compute_sequence_logps(
                policy_model,
                policy_batch
            )

        metrics = compute_dpo_loss(
            policy_logps["chosen"],
            policy_logps["rejected"],
            ref_logps["chosen"],
            ref_logps["rejected"],
            beta=DPO_BETA
        )

        loss = metrics["loss"]

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at batch {batch_idx}"
            )

        loss.backward()

        accum_count += 1
        accum_loss += loss.item()
        accum_accuracy += metrics[
            "preference_accuracy"
        ].item()

        global_step += 1

        is_last_batch = (
            batch_idx + 1 == len(hh_train_loader)
        )

        if accum_count == DPO_GRAD_ACCUM or is_last_batch:

            # Normalize gradients
            for param in policy_model.parameters():
                if param.grad is not None:
                    param.grad.div_(accum_count)

            grad_norm = torch.nn.utils.clip_grad_norm_(
                policy_model.parameters(),
                DPO_GRAD_CLIP
            )

            if not torch.isfinite(grad_norm):
                raise RuntimeError("Non-finite gradients")

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            optimizer_updates += 1

            avg_loss = accum_loss / accum_count
            avg_accuracy = accum_accuracy / accum_count

            hh_training_history.append({
                "epoch": epoch + 1,
                "update": optimizer_updates,
                "step": global_step,
                "loss": avg_loss,
                "preference_accuracy": avg_accuracy,
                "grad_norm": float(grad_norm.item())
            })

            progress.set_postfix({
                "loss": f"{avg_loss:.4f}",
                "accuracy": f"{avg_accuracy:.3f}",
                "updates": optimizer_updates
            })

            accum_count = 0
            accum_loss = 0.0
            accum_accuracy = 0.0

            # Periodic checkpoint
            if optimizer_updates % SAVE_EVERY_UPDATES == 0:
                save_dpo_checkpoint(optimizer_updates)

# Save final checkpoint
save_dpo_checkpoint(optimizer_updates)

elapsed_minutes = (time.time() - start_time) / 60

print("\nHH-RLHF DPO retraining completed!")
print("Optimizer updates:", optimizer_updates)
print("Expected updates:", total_updates)
print("Training minutes:", round(elapsed_minutes, 2))
print("Final loss:", hh_training_history[-1]["loss"])

assert optimizer_updates == total_updates

print("RETRAINING PASSED")


In [ ]:

import os
import shutil
from pathlib import Path

base = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/dpo/hh_rlhf"
)

disk = shutil.disk_usage("/kaggle/working")

print("Free disk space:",
      round(disk.free / 1024**3, 2), "GB")

print("\nCheckpoint status:")

for step in [100, 200, 300, 400]:
    folder = base / f"checkpoint-{step}"

    print(f"\nCheckpoint {step}:")

    if folder.exists():
        for file in folder.iterdir():
            if file.is_file():
                print(
                    file.name,
                    round(file.stat().st_size / 1024**3, 2),
                    "GB"
                )
    else:
        print("Not found")

print("\nTraining history exists:",
      (base / "training_history.json").exists())


In [ ]:

import torch
import shutil
from pathlib import Path

base = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/dpo/hh_rlhf"
)

checkpoint_300 = base / "checkpoint-300"

# Check disk space
disk = shutil.disk_usage("/kaggle/working")
print("Free disk space:", round(disk.free / 1024**3, 2), "GB")

# Verify model weights
assert (checkpoint_300 / "model.safetensors").is_file()
assert (checkpoint_300 / "config.json").is_file()

# Verify optimizer state can be deserialized
state = torch.load(
    checkpoint_300 / "training_state.pt",
    map_location="cpu",
    weights_only=True
)

print("Checkpoint update:", state["optimizer_updates"])
print("Processed batches:", state["global_step"])
print("Optimizer state entries:", len(state["optimizer"]["state"]))

assert state["optimizer_updates"] == 300
assert state["global_step"] == 9600

print("\nCheckpoint 300 verification: PASSED")

del state


In [ ]:

import shutil
from pathlib import Path

base = Path(
    "/kaggle/working/small-model-alignment-starter/"
    "outputs/dpo/hh_rlhf"
)

# Ensure checkpoint 300 is intact before cleanup
safe = base / "checkpoint-300"

assert (safe / "model.safetensors").is_file()
assert (safe / "training_state.pt").is_file()

# Remove older and incomplete checkpoints
for step in [100, 200, 400]:
    folder = base / f"checkpoint-{step}"

    if folder.exists():
        shutil.rmtree(folder)
        print(f"Removed checkpoint-{step}")

# Verify available disk space
disk = shutil.disk_usage("/kaggle/working")

print("\nFree disk space:",
      round(disk.free / 1024**3, 2), "GB")

print("Checkpoint 300 preserved:",
      (safe / "model.safetensors").exists())

print("\nDisk cleanup complete.")


In [ ]:

import gc
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM
from torch.optim import AdamW

checkpoint_300 = HH_OUTPUT_DIR / "checkpoint-300"

assert (checkpoint_300 / "model.safetensors").is_file()
assert (checkpoint_300 / "training_state.pt").is_file()

# Release existing policy and optimizer
if "optimizer" in globals():
    del optimizer
if "policy_model" in globals():
    del policy_model

gc.collect()
torch.cuda.empty_cache()

# Restore trained policy at update 300
policy_model = AutoModelForCausalLM.from_pretrained(
    str(checkpoint_300),
    dtype=torch.float32,
    local_files_only=True,
    low_cpu_mem_usage=True
).to(POLICY_DEVICE)

policy_model.config.use_cache = False
policy_model.gradient_checkpointing_enable()
policy_model.train()

# Restore optimizer
optimizer = AdamW(
    policy_model.parameters(),
    lr=DPO_LR,
    weight_decay=0.0
)

state = torch.load(
    checkpoint_300 / "training_state.pt",
    map_location="cpu",
    weights_only=True
)

optimizer.load_state_dict(state["optimizer"])

resume_update = int(state["optimizer_updates"])
resume_step = int(state["global_step"])

assert resume_update == 300
assert resume_step == 9600

optimizer.zero_grad(set_to_none=True)

# Reference model stays frozen
reference_model.eval()
reference_model.requires_grad_(False)

print("Policy restored from:", checkpoint_300)
print("Policy device:", next(policy_model.parameters()).device)
print("Reference device:", next(reference_model.parameters()).device)
print("Completed updates:", resume_update)
print("Completed batches:", resume_step)
print("Remaining updates:", total_updates - resume_update)
print("Optimizer restored:", len(optimizer.state))
print("\nCheckpoint restoration: PASSED")

del state


In [ ]:

import gc
import torch

# Release leftover computational graphs
for name in [
    "loss",
    "metrics",
    "policy_logps",
    "ref_logps",
    "policy_batch",
    "ref_batch",
    "batch",
    "progress"
]:
    if name in globals():
        del globals()[name]

# Clear gradients allocated before the OOM
if "optimizer" in globals():
    optimizer.zero_grad(set_to_none=True)

if "policy_model" in globals():
    policy_model.zero_grad(set_to_none=True)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print("GPU Memory Status")

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)

    print(f"\nGPU {i}")
    print("Free:", round(free / 1024**3, 2), "GB")
    print("Total:", round(total / 1024**3, 2), "GB")
    print(
        "PyTorch allocated:",
        round(torch.cuda.memory_allocated(i) / 1024**3, 2),
        "GB"
    )
    print(
        "PyTorch reserved:",
        round(torch.cuda.memory_reserved(i) / 1024**3, 2),
        "GB"
    )

print("\nMemory diagnostic completed.")


In [ ]:

import gc
import torch

# Clear remaining gradients
optimizer.zero_grad(set_to_none=True)
policy_model.zero_grad(set_to_none=True)

# Configure gradient checkpointing
policy_model.gradient_checkpointing_disable()
policy_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

policy_model.config.use_cache = False
policy_model.train()

# Ensure reference remains frozen
reference_model.eval()
reference_model.requires_grad_(False)

gc.collect()
torch.cuda.empty_cache()

# Test one small preference pair
test_pair = tokenize_dpo_pair(hh_train_pairs[0])
test_batch = dpo_collate_fn([test_pair])

try:
    with torch.no_grad():
        ref_logps = {}

        for pref in ["chosen", "rejected"]:
            ref_batch = {
                k: v.to(REFERENCE_DEVICE)
                for k, v in test_batch[pref].items()
            }
            ref_logps[pref] = compute_sequence_logps(
                reference_model, ref_batch
            ).to(POLICY_DEVICE)

    policy_logps = {}

    for pref in ["chosen", "rejected"]:
        policy_batch = {
            k: v.to(POLICY_DEVICE)
            for k, v in test_batch[pref].items()
        }
        policy_logps[pref] = compute_sequence_logps(
            policy_model, policy_batch
        )

    test_metrics = compute_dpo_loss(
        policy_logps["chosen"],
        policy_logps["rejected"],
        ref_logps["chosen"],
        ref_logps["rejected"],
        beta=DPO_BETA
    )

    test_metrics["loss"].backward()

    print("Test loss:", test_metrics["loss"].item())
    print("Backward pass: PASSED")

except torch.OutOfMemoryError:
    print("Backward pass: OUT OF MEMORY")

finally:
    optimizer.zero_grad(set_to_none=True)
    policy_model.zero_grad(set_to_none=True)

    for name in [
        "test_metrics", "policy_logps", "ref_logps",
        "policy_batch", "ref_batch", "test_batch", "test_pair"
    ]:
        globals().pop(name, None)

    gc.collect()
    torch.cuda.empty_cache()

print("\nGPU 0 peak allocated:",
      round(torch.cuda.max_memory_allocated(0) / 1024**3, 2),
      "GB")

In [ ]:

import gc
import json
import time
import random
import torch
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

assert resume_update == 300
assert resume_step == 9600

# Remaining training batches
remaining_batches = DPO_TRAIN_SIZE - resume_step

rng = random.Random(DPO_SEED + 300)
indices = rng.sample(
    range(len(hh_train_pairs)),
    remaining_batches
)
remaining_pairs = [hh_train_pairs[i] for i in indices]

resume_loader = DataLoader(
    DPODataset(remaining_pairs),
    batch_size=1,
    shuffle=False,
    collate_fn=dpo_collate_fn,
    num_workers=0
)

# Restore training history
history_path = HH_OUTPUT_DIR / "training_history.json"

with open(history_path, "r") as f:
    history = json.load(f)

hh_training_history = [
    row for row in history if row["update"] <= 300
]

optimizer_updates = 300
global_step = 9600

# Memory-efficient gradient checkpointing
policy_model.gradient_checkpointing_disable()
policy_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)
policy_model.config.use_cache = False
policy_model.train()
reference_model.eval()

optimizer.zero_grad(set_to_none=True)
torch.cuda.empty_cache()

start_time = time.time()

accum_count = 0
accum_loss = 0.0
accum_accuracy = 0.0

progress = tqdm(
    resume_loader,
    desc="HH-RLHF Resume"
)

for batch_idx, batch in enumerate(progress):

    with torch.no_grad():
        ref_logps = {}

        for pref in ["chosen", "rejected"]:
            ref_batch = {
                k: v.to(REFERENCE_DEVICE)
                for k, v in batch[pref].items()
            }
            ref_logps[pref] = compute_sequence_logps(
                reference_model, ref_batch
            ).to(POLICY_DEVICE)

    policy_logps = {}

    for pref in ["chosen", "rejected"]:
        policy_batch = {
            k: v.to(POLICY_DEVICE)
            for k, v in batch[pref].items()
        }
        policy_logps[pref] = compute_sequence_logps(
            policy_model, policy_batch
        )

    metrics = compute_dpo_loss(
        policy_logps["chosen"],
        policy_logps["rejected"],
        ref_logps["chosen"],
        ref_logps["rejected"],
        beta=DPO_BETA
    )

    loss = metrics["loss"]
    assert torch.isfinite(loss), "Non-finite loss"

    loss.backward()

    accum_count += 1
    accum_loss += loss.item()
    accum_accuracy += metrics["preference_accuracy"].item()
    global_step += 1

    is_last = batch_idx + 1 == len(resume_loader)

    if accum_count == DPO_GRAD_ACCUM or is_last:

        for p in policy_model.parameters():
            if p.grad is not None:
                p.grad.div_(accum_count)

        grad_norm = torch.nn.utils.clip_grad_norm_(
            policy_model.parameters(), DPO_GRAD_CLIP
        )
        assert torch.isfinite(grad_norm)

        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        optimizer_updates += 1

        hh_training_history.append({
            "update": optimizer_updates,
            "step": global_step,
            "loss": accum_loss / accum_count,
            "preference_accuracy": accum_accuracy / accum_count,
            "grad_norm": float(grad_norm.item())
        })

        progress.set_postfix({
            "loss": f"{accum_loss / accum_count:.4f}",
            "updates": optimizer_updates
        })

        accum_count = 0
        accum_loss = 0.0
        accum_accuracy = 0.0

        # Save model-only checkpoint at 400
        if optimizer_updates == 400:
            recovery_dir = HH_OUTPUT_DIR / "recovery-400"
            policy_model.save_pretrained(
                str(recovery_dir),
                safe_serialization=True
            )
            dpo_tokenizer.save_pretrained(str(recovery_dir))
            with open(history_path, "w") as f:
                json.dump(hh_training_history, f, indent=2)
            print("Recovery checkpoint 400 saved")

# Save final model
assert optimizer_updates == 469
assert global_step == 15000

HH_CHECKPOINT = HH_OUTPUT_DIR / "final"

policy_model.save_pretrained(
    str(HH_CHECKPOINT),
    safe_serialization=True
)
dpo_tokenizer.save_pretrained(str(HH_CHECKPOINT))

with open(history_path, "w") as f:
    json.dump(hh_training_history, f, indent=2)

print("\nHH-RLHF TRAINING COMPLETED")
print("Optimizer updates:", optimizer_updates)
print("Total batches:", global_step)
print("Time (minutes):", round((time.time() - start_time)/60, 2))
print("Final model:", HH_CHECKPOINT)


In [ ]:

from pathlib import Path
import zipfile
import shutil

HH_CHECKPOINT = HH_OUTPUT_DIR / "final"

archive_path = Path(
    "/kaggle/working/pa2_dpo_hh_rlhf_final.zip"
)

assert (HH_CHECKPOINT / "model.safetensors").is_file()
assert (HH_CHECKPOINT / "config.json").is_file()

# Archive only final model and training history
# to avoid filling Kaggle disk again.
files_to_archive = [
    p for p in HH_CHECKPOINT.rglob("*")
    if p.is_file()
]

history_path = HH_OUTPUT_DIR / "training_history.json"

if history_path.is_file():
    files_to_archive.append(history_path)

with zipfile.ZipFile(
    archive_path,
    "w",
    compression=zipfile.ZIP_STORED,
    allowZip64=True
) as zf:
    for file in files_to_archive:
        zf.write(
            file,
            arcname=file.relative_to(HH_OUTPUT_DIR)
        )

# Verify archive
with zipfile.ZipFile(archive_path) as zf:
    assert "final/model.safetensors" in zf.namelist()
    assert zf.testzip() is None

free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3

print("Archive created:", archive_path)
print("Archive size (GB):",
      round(archive_path.stat().st_size / 1024**3, 2))
print("Free disk space (GB):", round(free_gb, 2))
print("Archive verification: PASSED")


In [ ]:

import json
import torch
from tqdm.auto import tqdm

policy_model.eval()
reference_model.eval()

val_losses = []
val_accuracies = []
val_margins = []

with torch.inference_mode():
    for batch in tqdm(
        hh_val_loader,
        desc="HH-RLHF Validation"
    ):
        reference_logps = {}
        policy_logps = {}

        for pref in ["chosen", "rejected"]:
            ref_batch = {
                k: v.to(REFERENCE_DEVICE)
                for k, v in batch[pref].items()
            }
            pol_batch = {
                k: v.to(POLICY_DEVICE)
                for k, v in batch[pref].items()
            }

            reference_logps[pref] = compute_sequence_logps(
                reference_model, ref_batch
            ).to(POLICY_DEVICE)

            policy_logps[pref] = compute_sequence_logps(
                policy_model, pol_batch
            )

        metrics = compute_dpo_loss(
            policy_logps["chosen"],
            policy_logps["rejected"],
            reference_logps["chosen"],
            reference_logps["rejected"],
            beta=DPO_BETA
        )

        val_losses.append(metrics["loss"].item())
        val_accuracies.append(
            metrics["preference_accuracy"].item()
        )
        val_margins.append(
            metrics["reward_margin"].item()
        )

hh_validation_results = {
    "validation_pairs": len(val_losses),
    "mean_dpo_loss": sum(val_losses) / len(val_losses),
    "preference_accuracy": sum(val_accuracies) / len(val_accuracies),
    "mean_reward_margin": sum(val_margins) / len(val_margins)
}

with open(
    HH_OUTPUT_DIR / "validation_results.json",
    "w"
) as f:
    json.dump(hh_validation_results, f, indent=2)

print(json.dumps(hh_validation_results, indent=2))
print("\nHH-RLHF validation: PASSED")


In [ ]:

from datasets import load_dataset

# Load UltraFeedback preference pairs
ultra = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs"
)

print("Dataset loaded successfully!")
print("Total records:", len(ultra))
print("Columns:", ultra.column_names)

# Inspect the schema without printing the full conversation
sample = ultra[0]

for key, value in sample.items():
    print(
        key,
        "| type:", type(value).__name__,
        "| preview:", str(value)[:250]
    )

print("\nUltraFeedback dataset inspection: PASSED")


In [ ]:

import random
import json
from pathlib import Path

def extract_ultra_pair(row):
    chosen = row["chosen"]
    rejected = row["rejected"]

    if not chosen or not rejected:
        return None

    # Both conversations must end with assistant responses
    if chosen[-1]["role"] != "assistant":
        return None

    if rejected[-1]["role"] != "assistant":
        return None

    # Check shared conversation context
    chosen_context = chosen[:-1]
    rejected_context = rejected[:-1]

    if chosen_context != rejected_context:
        return None

    # Preserve complete multi-turn conversation
    # using the same serialization for both responses
    def serialize(messages):
        return "\n\n".join(
            f"{m['role'].capitalize()}: {m['content']}"
            for m in messages
        )

    prompt = serialize(chosen_context) + "\n\nAssistant:"
    chosen_response = chosen[-1]["content"].strip()
    rejected_response = rejected[-1]["content"].strip()

    if not chosen_response or not rejected_response:
        return None

    if chosen_response == rejected_response:
        return None

    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response
    }


# Extract and deduplicate
seen = set()
ultra_pairs = []

for row in ultra:
    pair = extract_ultra_pair(row)

    if pair is None:
        continue

    key = (
        pair["prompt"],
        pair["chosen"],
        pair["rejected"]
    )

    if key not in seen:
        seen.add(key)
        ultra_pairs.append(pair)

# Reproducible split
rng = random.Random(DPO_SEED)
rng.shuffle(ultra_pairs)

assert len(ultra_pairs) >= DPO_TRAIN_SIZE + DPO_VAL_SIZE

ultra_train_pairs = ultra_pairs[:DPO_TRAIN_SIZE]

ultra_val_pairs = ultra_pairs[
    DPO_TRAIN_SIZE:DPO_TRAIN_SIZE + DPO_VAL_SIZE
]

# Save split
ULTRA_OUTPUT_DIR = DPO_OUTPUT / "ultrafeedback"
ULTRA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

split_file = ULTRA_OUTPUT_DIR / "dataset_split.json"

with open(split_file, "w", encoding="utf-8") as f:
    json.dump({
        "train": ultra_train_pairs,
        "validation": ultra_val_pairs
    }, f, ensure_ascii=False)

print("Valid UltraFeedback pairs:", len(ultra_pairs))
print("Training pairs:", len(ultra_train_pairs))
print("Validation pairs:", len(ultra_val_pairs))
print("Split saved:", split_file)
print("\nUltraFeedback preparation: PASSED")


In [ ]:

import random
import json
from pathlib import Path

def extract_ultra_pair(row):
    chosen = row["chosen"]
    rejected = row["rejected"]

    if not chosen or not rejected:
        return None

    # Both conversations must end with assistant responses
    if chosen[-1]["role"] != "assistant":
        return None

    if rejected[-1]["role"] != "assistant":
        return None

    # Check shared conversation context
    chosen_context = chosen[:-1]
    rejected_context = rejected[:-1]

    if chosen_context != rejected_context:
        return None

    # Preserve complete multi-turn conversation
    # using the same serialization for both responses
    def serialize(messages):
        return "\n\n".join(
            f"{m['role'].capitalize()}: {m['content']}"
            for m in messages
        )

    prompt = serialize(chosen_context) + "\n\nAssistant:"
    chosen_response = chosen[-1]["content"].strip()
    rejected_response = rejected[-1]["content"].strip()

    if not chosen_response or not rejected_response:
        return None

    if chosen_response == rejected_response:
        return None

    return {
        "prompt": prompt,
        "chosen": chosen_response,
        "rejected": rejected_response
    }


# Extract and deduplicate
seen = set()
ultra_pairs = []

for row in ultra:
    pair = extract_ultra_pair(row)

    if pair is None:
        continue

    key = (
        pair["prompt"],
        pair["chosen"],
        pair["rejected"]
    )

    if key not in seen:
        seen.add(key)
        ultra_pairs.append(pair)

# Reproducible split
rng = random.Random(DPO_SEED)
rng.shuffle(ultra_pairs)

assert len(ultra_pairs) >= DPO_TRAIN_SIZE + DPO_VAL_SIZE

ultra_train_pairs = ultra_pairs[:DPO_TRAIN_SIZE]

ultra_val_pairs = ultra_pairs[
    DPO_TRAIN_SIZE:DPO_TRAIN_SIZE + DPO_VAL_SIZE
]

# Save split
ULTRA_OUTPUT_DIR = DPO_OUTPUT / "ultrafeedback"
ULTRA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

split_file = ULTRA_OUTPUT_DIR / "dataset_split.json"

with open(split_file, "w", encoding="utf-8") as f:
    json.dump({
        "train": ultra_train_pairs,
        "validation": ultra_val_pairs
    }, f, ensure_ascii=False)

print("Valid UltraFeedback pairs:", len(ultra_pairs))
print("Training pairs:", len(ultra_train_pairs))
print("Validation pairs:", len(ultra_val_pairs))
print("Split saved:", split_file)
print("\nUltraFeedback preparation: PASSED")
